# D1: codec-aware LoRA training of EVA-02

Gate 0 passed in all three folds: EVA-02 + LoRA loses 0.07 to 0.20 AUC from `resize224` to H.264 CRF 40 (results/gate0_compression_shift/). This notebook asks how much of the robustness that real-codec training gives back can a cheap codec prior recover, and whether the JPEG table matters or only the drop rate does (Phase 14b on the CNN: random >= table, real JPEG best, masking adds about +0.007 on top of it).

Every trained arm repeats Phase 13 run 2 exactly (EVA-02-L/14, LoRA on q and v, rank 16, alpha 32, 4 epochs, batch 32, lr 2e-4, AMP, DataParallel on both T4s, seed 42, data seed 42, 9,000 training frames per source corpus, head started from the frozen probe, best epoch on **source validation only**). The arms differ only in the train-time augmentation, which acts on the PIL face crop **before** the training transform; the evaluation transform is never augmented.

| Arm | Train-time augmentation | What it answers |
| --- | --- | --- |
| `plain` | none: the Phase 13 run 2 checkpoints, re-scored (no training) | the baseline; must reproduce Gate 0 (PASS within 0.002) |
| `realh264` | p = 0.5: resize the crop to 224x224 (bilinear, as Gate 0), one-frame libx264 clip at CRF uniform in {23..40} (preset veryfast, yuv420p, 1 thread), decode | the real-codec upper reference |
| `qmask` | p = 1: JPEG-luminance-table-weighted 8x8 block-DCT coefficient dropout at native crop resolution, max drop 0.35, DC kept (codec7 tables) | the codec prior |
| `randmask` | as `qmask` with the uniform table at the same mean drop rate | is it the table or the rate? |
| `realjpeg` | p = 0.5: JPEG re-save at native resolution, quality uniform in [20, 95] (as Phase 14b) | real-JPEG reference |
| `randmask_realh264` | `randmask`, then `realh264` | does masking add to real codec training? |
| `qmask_realh264` | `qmask`, then `realh264` | the same with the table |
| `invmask` | inverted table at the same mean | does the codec ordering matter? |
| `noaug` | none, but retrained in this notebook like the other arms | run-to-run noise: `noaug - plain` measures it directly |

Masks: the crop is edge-padded to a multiple of 8, each RGB channel's 8x8 blocks lose the drawn coefficients (one keep mask per image, as codec7's `BlockDCTMask`), then the crop is cut back and clipped to [0, 255]. Randomness comes from Python's `random`, which torch reseeds in every DataLoader worker, so no two workers draw the same masks.

**Evaluation, identical to Gate 0** (so the numbers are comparable with results/gate0_compression_shift/): each fold's Gate 0 subset (up to 500 real + 500 fake held-out videos, seed 42; DFDC all 381, 8 frames each) under `clean`, `resize224`, `jpeg50`, `jpeg20`, `h264_crf23`, `h264_crf40` (each video's frames resized to 224 px, one libx264 clip at preset medium, decoded). Video score = mean frame probability. The H.264 clips are built once per fold under /tmp and shared by every arm (about 1.2 GB per CRF for Celeb-DF and FF++), then deleted.

**Statistics (Cell 10):** paired cluster bootstrap (1,000 resamples) per fold and condition for every arm against `plain`, `qmask` against `randmask` and `invmask`, the two mask + real-H.264 arms against `realh264`, and `realh264` against `realjpeg`. Gap-closure ratio = (mask arm - plain) / (realh264 - plain), and the same with `realjpeg`, at `h264_crf40` and `jpeg20`; it is reported only when the denominator's interval excludes zero, otherwise "not defined". The bootstrap resamples test videos only. `plain` was trained in an earlier session and every other arm is retrained here (AMP and DataParallel are not bit-reproducible), so each delta also carries training noise: the EVA-02 seed SD is about 0.006 (Phase 13b, DFDC fold, seeds 42/43/44), and `paired_deltas.csv` flags `beyond_seed_noise` (|delta| >= 0.012). Read a delta below that as not established even when its interval excludes zero.

**Inputs to attach** (right panel, **+ Add Input**):
1. The three face-crop datasets: Celeb-DF Preprocessed, Faceforensics++ Videos Cropped Faces, dfdc faces of the train sample.
2. `phase13-eva02-output`: `faculty_table_eva02_l14.csv` with `features_eva02_l14.npy` and `bank_eva02_l14.csv` (the frozen-probe features are reused, saving about 20 min).
3. `phase13-eva02-lora`: `lora_celebdf_best.pt`, `lora_ffpp_best.pt`, `lora_dfdc_best.pt` (Phase 13 run 2; the `plain` arm). If they sit next to the table in `phase13-eva02-output`, that copy is used.
4. `phase11-checkpoints`: not read by this notebook (Cell 2 only defines the class). Keep it only if it is already attached; attaching or removing inputs re-syncs all of them.
5. Optional, from the second session on: the output of an earlier version of this notebook. Every (arm, fold) whose `done.json` signature matches is copied instead of retrained.

**Settings:** GPU T4 x2, Internet ON (the EVA-02 base weights download from Hugging Face).

**How to run**
1. Keep `SMOKE = True` in Cell 6 → **Save Version → Save & Run All (Commit)**. About 45 to 60 min: `plain` on every fold, every trained arm for 1 epoch on the first fold only (this covers every code path), smoke frames, H.264 subsets of 12 + 12 videos, 200 bootstrap resamples, the plain consistency check, a forced gap-closure ratio (path test), the summary and the zip. Every Cell 7 line must read `PASS`. If a unit fails, the others still run, the summary and the zip are written, and Cell 10 then raises, so the commit shows red with every traceback. The smoke AUCs are not results.
2. Set `SMOKE = False` → **Save & Run All (Commit)**. Cell 7 prints the projected hours per (arm, fold) and which arms fit in this session. Arms run in the priority order of `ARMS`; a unit that would cross 11 h is skipped with a log line, and the summary and zip are always written.
3. Download `d1.zip` from the version's Output tab. For the arms that did not fit, attach that version's output to a new version and commit again: finished units are reused.

**Expected time:** one EVA-02 LoRA fold took 0.78 to 0.84 h on T4 x2 (Phase 13b); scoring six conditions is about 0.2 h for Celeb-DF and FF++ (8,000 frames at about 75 img/s) and about 0.1 h for DFDC; the H.264 test clips take about 4 min per CRF per fold, once. So about 1.1 h per trained arm per fold and about 3.3 h per arm: session 1 fits `plain` and about three trained arms (`realh264`, `qmask`, `randmask`, the gap-closure set); the remaining five (`noaug` last) need two more sessions. Move `noaug` up in `ARMS` to run the run-noise control before `invmask`. `realh264` adds a CPU encode per augmented sample: Cell 7 measures the augmented loader (idle machine, scaled by 3/4 for the CPU the training process takes) and uses it in the projection if it is slower than the GPU. After each finished unit the projection is corrected by measured / projected time, separately for the real-H.264 arms and the others.

**Outputs** (`/kaggle/working/d1`): `auc.csv` (arm, fold, condition, AUC, videos; rewritten after every condition), `pred_<arm>_<fold>_<condition>.csv`, `<arm>/<fold>/` (training history, `lora_<fold>.json`, `done.json`; `lora_<fold>_best.pt` stays in the version output, not in the zip), `consistency.csv`, `status.csv`, `plan.csv`, `preflight_d1.json`, `auc_wide.csv`, `auc_mean_over_folds.csv`, `paired_deltas.csv`, `paired_deltas_mean.csv`, `gap_closure.csv`, `d1_conditions.png`, `run_metadata.json`, and `d1.zip`.

## Cell 1: Phase 11 setup (known dataset paths)

In [ ]:
import os, re, json, time, random, warnings, itertools
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, roc_curve, f1_score, accuracy_score, \
                            precision_score, recall_score, confusion_matrix
from PIL import Image
import timm

warnings.filterwarnings("ignore")

# ===================== CONFIG ========================================
TERM  = 0    # 1=audit(no GPU) 2=matrix+LOO 3=GradCAM 4=robustness 11=dual/DANN/CAM
SMOKE = False    # False = the real run

SEED, IMG_SIZE, BATCH = 42, 224, 64
LR, WD, NUM_WORKERS = 1e-4, 1e-4, 4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if SMOKE:
    EPOCHS, PATIENCE = 1, 1
    MAX_IMGS_PER_DATASET, MAX_TEST_IMGS, MAX_VAL_IMGS = 3000, 3000, 1500
else:
    EPOCHS, PATIENCE = 20, 4
    MAX_IMGS_PER_DATASET, MAX_TEST_IMGS, MAX_VAL_IMGS = 30000, None, None

OUT = Path("/kaggle/working/phase5_results"); OUT.mkdir(parents=True, exist_ok=True)
PHASE4_DIR = Path("/kaggle/input/phase4-results")

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ===================== DISCOVERY =====================================
INPUT_ROOT = Path("/kaggle/input")

def _find_root(*hints, max_depth=5):
    cands = []
    for dp, dns, fns in os.walk(INPUT_ROOT):
        rel = os.path.relpath(dp, INPUT_ROOT).lower()
        if rel == ".": continue
        if rel.count(os.sep) + 1 > max_depth:
            dns[:] = []; continue
        if any(h in rel for h in hints):
            cands.append((rel.count(os.sep) + 1, dp)); dns[:] = []
    return Path(min(cands)[1]) if cands else None

# Known paths first: searching /kaggle/input takes up to an hour in a commit.
_KNOWN = {"celebdf": "/kaggle/input/datasets/amanrawat001/celeb-df-preprocessed",
          "ffpp":    "/kaggle/input/datasets/debajyatidey/faceforensics-videos-cropped-faces",
          "dfdc":    "/kaggle/input/datasets/itamargr/dfdc-faces-of-the-train-sample"}
def _root(name, *hints):
    p = Path(_KNOWN[name])
    return p if p.is_dir() else _find_root(*hints)
CELEBDF_ROOT = _root("celebdf", "celebdf", "celeb-df", "celeb_df")
FFPP_ROOT    = _root("ffpp", "faceforensics", "ff++", "ffpp", "forensics")
DFDC_ROOT    = _root("dfdc", "dfdc")
print("Celeb-DF:", CELEBDF_ROOT, "\nFF++    :", FFPP_ROOT, "\nDFDC    :", DFDC_ROOT)

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
CLASS_DIRS = {"fake","fakes","real","reals","original","originals","manipulated",
              "manipulated_sequences","original_sequences","pristine","youtube",
              "train","val","valid","validation","test"}

def _walk_images(root):
    for dp, _, fns in os.walk(root):
        for fn in fns:
            if fn.lower().endswith(IMG_EXT):
                yield os.path.join(dp, fn)

def _label_from_path(path):
    for p in reversed([x.lower() for x in Path(path).parts]):
        if p in ("fake","fakes","manipulated","manipulated_sequences","deepfake"): return 1
        if p in ("real","reals","original","originals","pristine","youtube",
                 "original_sequences"): return 0
    return None

_FRAME_RE = re.compile(r"^(.*?)[_-]?frame[_-]?\d+", re.IGNORECASE)

def vid_celebdf(path):
    stem = Path(path).stem
    m = _FRAME_RE.match(stem)
    return m.group(1) if (m and m.group(1)) else "_".join(stem.split("_")[:3])

def vid_ffpp(path):
    """fake/588_556_crops/frame_0019.png -> the FOLDER is the video."""
    p = Path(path)
    return p.parent.name if p.parent.name.lower() not in CLASS_DIRS else p.stem

def vid_dfdc(path):
    """aapnvogymq_100_1.png -> aapnvogymq"""
    return Path(path).stem.split("_")[0]

PARSERS = {"celebdf": (CELEBDF_ROOT, vid_celebdf),
           "ffpp":    (FFPP_ROOT,    vid_ffpp),
           "dfdc":    (DFDC_ROOT,    vid_dfdc)}

def _ffpp_identity_groups(videos):
    """Tie fake 588_556 to real 588 and real 556 — same face, same split."""
    parent = {}
    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb: parent[ra] = rb
    nums_of = {}
    for v in videos:
        nums = re.findall(r"\d+", v.split("::")[-1])
        nums_of[v] = nums
        for n in nums[1:]: union(nums[0], n)
    return {v: (f"ffppid::{find(ns[0])}" if ns else v) for v, ns in nums_of.items()}

def build_manifest(name):
    root, vid_fn = PARSERS[name]
    assert root is not None, f"{name} not attached"
    rows = []
    for p in _walk_images(root):
        y = _label_from_path(p)
        if y is None: continue
        rows.append((p, y, f"{name}::{vid_fn(p)}", name))
    df = pd.DataFrame(rows, columns=["path","label","video","dataset"])
    assert len(df) > 0, f"No labelled images under {root}"
    if name == "ffpp":
        df["group"] = df.video.map(_ffpp_identity_groups(df.video.unique()))
        print(f"  [ffpp] {df.video.nunique():,} videos -> {df.group.nunique():,} identity groups")
    else:
        df["group"] = df.video
    return df

def split_by_video(df, test_size=0.15, val_size=0.15, seed=SEED):
    n = df.group.nunique()
    assert n >= 20, f"Only {n} groups — parser broken: {df.group.head(3).tolist()}"
    gss = GroupShuffleSplit(1, test_size=test_size, random_state=seed)
    tr_idx, te_idx = next(gss.split(df, groups=df.group))
    trv, te = df.iloc[tr_idx].reset_index(drop=True), df.iloc[te_idx].reset_index(drop=True)
    gss2 = GroupShuffleSplit(1, test_size=val_size/(1-test_size), random_state=seed)
    tr_i2, va_i = next(gss2.split(trv, groups=trv.group))
    tr, va = trv.iloc[tr_i2].reset_index(drop=True), trv.iloc[va_i].reset_index(drop=True)
    for a, b, nm in [(tr,va,"train/val"), (tr,te,"train/test"), (va,te,"val/test")]:
        for col in ("group", "video"):
            ov = set(a[col]) & set(b[col])
            assert len(ov) == 0, f"LEAK in {nm} on {col}: {len(ov)} shared"
    return tr, va, te

def cap_per_dataset(df, cap=None, seed=SEED):
    cap = MAX_IMGS_PER_DATASET if cap is None else cap
    out = [g.sample(min(len(g), cap), random_state=seed) if len(g) > cap else g
           for _, g in df.groupby("dataset")]
    return pd.concat(out).sample(frac=1.0, random_state=seed).reset_index(drop=True)

def subsample(df, cap, seed=SEED):
    """Shrink a split by dropping whole VIDEOS, never single frames."""
    if cap is None or len(df) <= cap: return df
    vids = df.groupby("video").label.first()
    frac, keep = cap/len(df), []
    for lbl, grp in vids.groupby(vids):
        n = max(3, int(round(len(grp)*frac)))
        keep += list(pd.Series(grp.index).sample(min(n, len(grp)), random_state=seed))
    return df[df.video.isin(keep)].reset_index(drop=True)

def create_model(pretrained=True):
    return timm.create_model("efficientnet_b0", pretrained=pretrained, num_classes=1)


def load_checkpoint(model, path):
    ck = torch.load(path, map_location="cpu")
    for key in ("ema","model_ema","state_dict","model"):
        if isinstance(ck, dict) and key in ck and isinstance(ck[key], dict):
            ck = ck[key]; break
    ck = {k.replace("module.",""): v for k, v in ck.items()}
    # Phase 11 saves SingleNet / DualNet wrappers, whose keys carry a "net."
    # prefix. Without this strip every key mismatches, strict=False loads
    # NOTHING, and a randomly-initialised model gets scored as if it were real.
    if ck and all(k.startswith("net.") for k in ck):
        ck = {k[4:]: v for k, v in ck.items()}
    miss, unexp = model.load_state_dict(ck, strict=False)
    n_total = len(model.state_dict()); n_loaded = n_total - len(miss)
    if miss or unexp: print(f"  [load] missing={len(miss)} unexpected={len(unexp)}")
    if n_loaded < 0.9 * n_total:
        raise RuntimeError(
            f"{Path(path).name}: only {n_loaded}/{n_total} tensors loaded. "
            f"Refusing to score a model that did not load.")
    return model

print(f"\nTERM={TERM} | SMOKE={SMOKE} | epochs={EPOCHS} | device={DEVICE}")

# ===================== TERM 1: DATA AUDIT ============================
if TERM == 1:
    AUDIT = {}
    for name in ["celebdf", "ffpp", "dfdc"]:
        print("=" * 70)
        if PARSERS[name][0] is None:
            print(f"{name.upper()}: NOT FOUND"); continue
        df = build_manifest(name)
        fpv = len(df) / df.video.nunique()
        vl = df.groupby("video").label.first()
        print(f"{name.upper()}: {len(df):,} imgs | {df.video.nunique():,} videos "
              f"| {df.group.nunique():,} groups | {fpv:.1f} frames/video")
        print(f"  videos: real {int((vl==0).sum()):,} / fake {int((vl==1).sum()):,}")
        print("  OK" if fpv >= 5 else "  !! FAIL: parser not grouping frames")
        tr, va, te = split_by_video(df)
        AUDIT[name] = dict(images=len(df), videos=int(df.video.nunique()),
                           groups=int(df.group.nunique()), fpv=round(fpv,1))
    print(pd.DataFrame(AUDIT).T.to_string())
    print("\nTERM 1 done.")


## Cell 2: Phase 11 dual-stream definitions

Unused by D1; Cell 1 and Cell 3 refer to them, so they stay.

In [ ]:
# Definitions only: no old CAM/DANN/dual-stream training loops.
import math
import torch.nn.functional as F
from torchvision import transforms as T
MEAN, STD = (0.485,0.456,0.406), (0.229,0.224,0.225)
eval_tf = T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN,STD)])

def _dct_mat(n):
    k = torch.arange(n, dtype=torch.float64); i = torch.arange(n, dtype=torch.float64)
    D = torch.cos(math.pi * (2*i[None,:] + 1) * k[:,None] / (2*n))
    D *= math.sqrt(2.0/n); D[0] /= math.sqrt(2.0)
    return D.float()


class FAD(nn.Module):
    """Whole-image DCT -> additive learnable band masks -> inverse DCT.
    Bands come back in IMAGE space, so the frequency stream stays aligned
    with RGB and an ImageNet-pretrained stem still transfers."""
    def __init__(self, size=IMG_SIZE, edges=(0.0, 1/16, 1/8, 1.0)):
        super().__init__()
        self.n_bands = len(edges) - 1
        self.register_buffer("D", _dct_mat(size), persistent=False)
        u = torch.arange(size, dtype=torch.float32)
        r = (u[:,None] + u[None,:]) / (2*size - 2)
        self.register_buffer("base", torch.stack([
            ((r >= lo) & ((r < hi) if hi < 1.0 else (r <= hi))).float()
            for lo, hi in zip(edges[:-1], edges[1:])]), persistent=False)
        self.delta = nn.Parameter(torch.zeros(self.n_bands, size, size))
    def forward(self, x01):
        D = self.D
        X = D @ x01 @ D.transpose(0,1)
        m = self.base + torch.tanh(self.delta)
        X = X.unsqueeze(1) * m[None,:,None]
        return (D.transpose(0,1) @ X @ D).flatten(1,2)


def _feat_dim(m):
    was = m.training; m.eval()
    with torch.no_grad():
        f = m.forward_features(torch.zeros(1, 3, IMG_SIZE, IMG_SIZE,
                               device=next(m.parameters()).device))
    m.train(was)
    return f.shape[1]


class DualNet(nn.Module):
    """Takes ImageNet-NORMALISED input and returns (N,1), so every existing
    helper (predict / score / video_level) works on it unchanged.  The
    [0,1] image needed by the DCT is recovered internally."""
    def __init__(self, pretrained=False):
        super().__init__()
        self.rgb = create_model(pretrained=pretrained)
        assert hasattr(self.rgb, "forward_features"), \
            "create_model() must expose forward_features (timm backbone)"
        self.fad  = FAD()
        self.freq = timm.create_model("efficientnet_b0", pretrained=pretrained,
                                      num_classes=0, in_chans=3*self.fad.n_bands)
        self.register_buffer("mean", torch.tensor(MEAN).view(1,3,1,1))
        self.register_buffer("std",  torch.tensor(STD).view(1,3,1,1))
        d_r = _feat_dim(self.rgb); d_f = self.freq.num_features
        self.head      = nn.Sequential(nn.Dropout(0.2), nn.Linear(d_r + d_f, 1))
        self.head_rgb  = nn.Linear(d_r, 1)
        self.head_freq = nn.Linear(d_f, 1)
        self._aux, self._feat, self._pooled = None, None, None
    def forward(self, xn):
        fr = self.rgb.forward_features(xn); self._feat = fr
        zr = F.adaptive_avg_pool2d(fr, 1).flatten(1); self._pooled = zr
        x01 = (xn * self.std + self.mean).clamp(0, 1)
        with torch.autocast('cuda', enabled=False):
            fb = self.fad(x01.float())          # DC term is large: not fp16-safe
        fb = ((fb - fb.mean(dim=(2, 3), keepdim=True))
              / (fb.std(dim=(2, 3), keepdim=True) + 1e-5))
        zf = self.freq(fb.to(xn.dtype))
        self._aux = (self.head_rgb(zr).squeeze(1), self.head_freq(zf).squeeze(1))
        return self.head(torch.cat([zr, zf], 1))          # (N,1)


print("Original FAD and DualNet definitions loaded; pretrained downloads disabled.")


## Cell 3: Phase 13 library (definitions only)

Unchanged. D1 uses `fold_frames` (the exact frames), `run_probes`, `run_lora_fold`, `build_detector` and `preflight` from it.

In [ ]:
"""Phase 13: CLIP ViT-L/14, DINOv2 ViT-g/14 and EVA-02-L/14 with LoRA on the
attention projections, evaluated leave-one-dataset-out.

Two arms per backbone and held-out corpus, scored on the same frames:
  probe  frozen backbone, cached features, logistic-regression head
  lora   low-rank updates on the attention projections plus a linear head.
         The head starts from the probe (LP-FT), so epoch 0 equals the probe.
Checkpoints are chosen on source validation only. The held-out corpus is
scored once, after training. Definitions only: nothing trains on import.
"""
import faulthandler
import gc
import itertools
import json
import math
import os
import random
import shutil
import sys
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image
from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
import timm
from timm.data import resolve_model_data_config


CORPORA = ('celebdf', 'ffpp', 'dfdc')
BACKBONES = {
    'clip_l14': dict(
        timm_id='vit_large_patch14_clip_224.openai', kwargs={},
        note='CLIP ViT-L/14 (OpenAI), image-text contrastive pretraining, 304M params'),
    'dinov2_g14': dict(
        timm_id='vit_giant_patch14_dinov2.lvd142m', kwargs={'img_size': 224}, half=True,
        note='DINOv2 ViT-g/14, self-distillation on LVD-142M, 1.1B params; '
             'position embeddings resampled from 518 to 224; frozen weights stored in fp16'),
    'eva02_l14': dict(
        timm_id='eva02_large_patch14_224.mim_m38m', kwargs={},
        note='EVA-02-L/14, masked image modelling on Merged-38M, 304M params'),
}
BANK_COLS = ['path', 'dataset', 'video_id', 'group_id', 'label', 'split']
CKPT_OK = {}   # backbone -> gradient checkpointing passed the unit test

# Kaggle's T4 x2 can hang in DataParallel's GPU-to-GPU copies; route them through the host.
os.environ.setdefault('NCCL_P2P_DISABLE', '1')
# In a commit, 90 silent minutes means a hang: dump every thread's stack and exit, so the
# run fails with a traceback instead of idling to the 12 h limit. Off in interactive sessions.
COMMIT = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') == 'Batch'
WATCHDOG_MIN = 90

try:   # kernel start: Kaggle's 12 h limit counts from the session start, not from this cell
    import psutil
    T0 = psutil.Process(os.getpid()).create_time()
except Exception:
    T0 = time.time()


def hours():
    return (time.time() - T0) / 3600


def log(msg):
    print(f'[{hours():6.2f}h] {msg}', flush=True)
    if COMMIT:
        try:
            faulthandler.dump_traceback_later(WATCHDOG_MIN * 60, exit=True, file=sys.__stderr__)
        except Exception:
            pass


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def jsonable(x):
    if isinstance(x, dict):
        return {str(k): jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [jsonable(v) for v in x]
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, Path):
        return str(x)
    return x


# ---------------------------------------------------------------- manifest

def validate_manifest(frame):
    required = {'path', 'dataset', 'video_id', 'group_id', 'label', 'split'}
    if required - set(frame):
        raise ValueError(f'Missing manifest columns: {required - set(frame)}')
    d = frame.copy()
    if d[list(required)].isna().any().any():
        raise ValueError('Manifest contains missing values')
    for col in ['dataset', 'video_id', 'group_id', 'split']:
        d[col] = d[col].astype(str).str.strip()
        if d[col].eq('').any():
            raise ValueError(f'Empty {col}')
    if not set(d.dataset) <= set(CORPORA):
        raise ValueError('Use canonical dataset names: celebdf, ffpp, dfdc')
    if not set(d.split) <= {'train', 'val', 'test'}:
        raise ValueError('Use existing train/val/test assignments')
    if not set(d.label) <= {0, 1}:
        raise ValueError('Labels must be 0=real, 1=fake')
    d['path'] = d.path.map(os.path.normpath)   # string only: touching 230k files costs hours in a commit
    if d.path.duplicated().any():
        raise ValueError('Duplicate frame paths; use one row per file')
    if (d.groupby(['dataset', 'video_id']).label.nunique() > 1).any():
        raise ValueError('A video has conflicting labels')
    if (d.groupby(['dataset', 'video_id']).group_id.nunique() > 1).any():
        raise ValueError('A video has multiple identity/source groups')
    for key in ['video_id', 'group_id']:
        if (d.groupby(['dataset', key]).split.nunique() > 1).any():
            raise ValueError(f'{key} leakage across original splits')
    # Paths come from a directory listing; spot-check a sample instead of every file.
    missing = [p for p in d.path.sample(min(len(d), 200), random_state=0) if not os.path.isfile(p)]
    if missing:
        raise FileNotFoundError(f'{len(missing)} of 200 sampled frames are missing; first: {missing[0]}')
    return d


def diverse_cap(d, cap, seed):
    """Shuffle, then take one frame per video before any video gets a second."""
    d = d.sample(frac=1, random_state=seed).copy()
    d['_round'] = d.groupby(['dataset', 'video_id']).cumcount()
    return d.sort_values('_round', kind='stable').head(cap).drop(columns='_round')


def balanced_cap(d, cap, seed):
    return pd.concat([diverse_cap(d[d.label == y], cap // 2, seed) for y in (0, 1)])


def first_frames(d, k):
    return d.sort_values('path').groupby(['dataset', 'video_id'], group_keys=False).head(k)


def limit_videos(d, n, seed):
    """Smoke mode: at most n whole videos per (dataset, label)."""
    keep = (d[['dataset', 'video_id', 'label']].drop_duplicates()
            .sample(frac=1, random_state=seed).groupby(['dataset', 'label']).head(n))
    return d.merge(keep[['dataset', 'video_id']], on=['dataset', 'video_id'])


def _ids(frame, col):
    return set(zip(frame.dataset, frame[col]))


def fold_frames(d, held, cfg):
    """Train/val come from the two source corpora; test is every video of `held`.
    Frames are drawn with `data_seed`, so a new training seed keeps the same frames."""
    sources = [c for c in CORPORA if c != held]
    data_seed = cfg.get('data_seed', cfg['seed'])
    tr = d[d.dataset.isin(sources) & d.split.eq('train')]
    va = d[d.dataset.isin(sources) & d.split.eq('val')]
    te = d[d.dataset.eq(held)]
    if cfg['smoke']:
        tr, va, te = (limit_videos(x, cfg['smoke_videos'], data_seed) for x in (tr, va, te))
    train = pd.concat([balanced_cap(tr[tr.dataset.eq(s)], cfg['train_per_source'], data_seed)
                       for s in sources])
    val = first_frames(va, cfg['eval_frames_per_video'])
    test = first_frames(te, cfg['eval_frames_per_video'])
    for name, part, need in [('train', train, sources), ('val', val, sources), ('test', test, [held])]:
        for ds in need:
            if part[part.dataset.eq(ds)].label.nunique() != 2:
                raise ValueError(f'{held} fold: {name}/{ds} needs both classes')
    for col in ('video_id', 'group_id'):
        if _ids(train, col) & _ids(val, col):
            raise ValueError(f'{held} fold: train and val share a {col}')
    if held in set(train.dataset) | set(val.dataset):
        raise ValueError(f'{held} fold: held-out corpus leaked into training')
    return [x.reset_index(drop=True) for x in (train, val, test)]


# -------------------------------------------------------------------- data

class Frames(Dataset):
    def __init__(self, frame, tf):
        self.paths = frame.path.tolist()
        self.labels = frame.label.astype(float).tolist()
        self.tf = tf

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            x = self.tf(im.convert('RGB'))
        return x, self.labels[i], i


def loader(frame, tf, cfg, train=False):
    return DataLoader(Frames(frame, tf),
                      batch_size=cfg['batch_size'] if train else cfg['eval_batch_size'],
                      shuffle=train, drop_last=train, num_workers=cfg['workers'],
                      timeout=600 if cfg['workers'] else 0,   # a stuck worker raises instead of hanging
                      pin_memory=True, generator=torch.Generator().manual_seed(cfg['seed']))


def backbone_transforms(model, size):
    """Each backbone keeps its own pretraining normalisation; faces are resized, not cropped."""
    dc = resolve_model_data_config(model)
    resize = T.Resize((size, size), interpolation=T.InterpolationMode.BICUBIC, antialias=True)
    tail = [T.ToTensor(), T.Normalize(dc['mean'], dc['std'])]
    return (T.Compose([T.RandomHorizontalFlip(), resize] + tail),
            T.Compose([resize] + tail),
            dict(mean=list(dc['mean']), std=list(dc['std'])))


# -------------------------------------------------------------------- LoRA

class LoRAAdapter(nn.Module):
    """s * B A x with B = 0 at the start, so the wrapped layer is unchanged until trained."""
    def __init__(self, d_in, d_out, r, alpha, dropout):
        super().__init__()
        self.A = nn.Parameter(torch.empty(r, d_in))
        self.B = nn.Parameter(torch.zeros(d_out, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.scale = alpha / r
        self.drop = nn.Dropout(dropout)
        self.calls = 0

    def forward(self, x):
        self.calls += 1
        return F.linear(F.linear(self.drop(x), self.A.to(x.dtype)), self.B.to(x.dtype)) * self.scale


class LoRALinear(nn.Module):
    """Frozen nn.Linear plus a low-rank update: W x + b + s B A x."""
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.lora = LoRAAdapter(base.in_features, base.out_features, r, alpha, dropout)

    def forward(self, x):
        return self.base(x) + self.lora(x)


class LoRAFusedQKV(nn.Module):
    """Fused qkv projection; low-rank updates on the chosen q/k/v slices only."""
    def __init__(self, base, parts, r, alpha, dropout):
        super().__init__()
        assert base.out_features % 3 == 0
        self.base, self.d = base, base.out_features // 3
        self.lora = nn.ModuleDict({p: LoRAAdapter(base.in_features, self.d, r, alpha, dropout)
                                   for p in parts})

    def forward(self, x):
        out = self.base(x)
        delta = [self.lora[p](x) if p in self.lora else out.new_zeros(*out.shape[:-1], self.d)
                 for p in 'qkv']
        return out + torch.cat(delta, -1).to(out.dtype)


def inject_lora(model, targets, r, alpha, dropout):
    """Freeze the model and wrap every attention block's projections in place.

    timm ViTs (CLIP, DINOv2) fuse q/k/v into one `qkv` Linear; EVA-02 has
    q_proj/k_proj/v_proj. Target `o` is the attention output projection `proj`.
    """
    if not targets or not set(targets) <= set('qkvo'):
        raise ValueError("lora_targets must be letters from 'qkvo', e.g. 'qv'")
    for p in model.parameters():
        p.requires_grad_(False)
    wrapped = []
    for name, m in list(model.named_modules()):
        if isinstance(getattr(m, 'qkv', None), nn.Linear):
            parts = ''.join(p for p in 'qkv' if p in targets)
            if parts:
                m.qkv = LoRAFusedQKV(m.qkv, parts, r, alpha, dropout)
        elif all(isinstance(getattr(m, f'{p}_proj', None), nn.Linear) for p in 'qkv'):
            for p in 'qkv':
                if p in targets:
                    setattr(m, f'{p}_proj', LoRALinear(getattr(m, f'{p}_proj'), r, alpha, dropout))
        else:
            continue
        if 'o' in targets:
            if not isinstance(getattr(m, 'proj', None), nn.Linear):
                raise TypeError(f'{name}: no attention output projection to wrap')
            m.proj = LoRALinear(m.proj, r, alpha, dropout)
        wrapped.append(name)
    if len(wrapped) != len(model.blocks):
        raise RuntimeError(f'Wrapped {len(wrapped)} attention modules, model has {len(model.blocks)} blocks')
    return wrapped


def adapters(model):
    return [m for m in model.modules() if isinstance(m, LoRAAdapter)]


def set_checkpointing(backbone, on):
    backbone.set_grad_checkpointing(on)
    if on and getattr(backbone, '_p13_hook', None) is None:
        # Reentrant checkpointing needs an input that requires grad. The frozen
        # patch embedding produces none, and without this LoRA gets no gradient.
        backbone._p13_hook = backbone.patch_embed.register_forward_hook(
            lambda m, i, o: o.requires_grad_(True) if torch.is_grad_enabled() else o)


# ------------------------------------------------------------------ models

def create_backbone(key, pretrained=True, **override):
    spec = BACKBONES[key]
    try:
        model = timm.create_model(spec['timm_id'], pretrained=pretrained, num_classes=0,
                                  **{**spec['kwargs'], **override})
    except Exception as e:
        raise RuntimeError(f'Could not create {spec["timm_id"]} (timm {timm.__version__}). '
                           f'Check that Internet is ON in the notebook settings. {e}') from e
    # fp16 frozen weights halve the 1.1B model's memory. LoRA weights, added after this, stay fp32.
    return model.half() if pretrained and spec.get('half') else model


class Encoder(nn.Module):
    """Pooled features in fp32. Autocast sits inside forward so DataParallel replicas use it."""
    def __init__(self, backbone, amp=True):
        super().__init__()
        self.backbone, self.amp = backbone, amp
        self.in_dtype = next(backbone.parameters()).dtype

    def forward(self, x):
        if self.amp and x.is_cuda:
            with torch.autocast('cuda', dtype=torch.float16):
                return self.backbone(x).float()
        return self.backbone(x.to(self.in_dtype)).float()


class Detector(nn.Module):
    def __init__(self, encoder, dim):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(dim, 1)
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
        return self.head(self.encoder(x)).squeeze(1)


def parallel(module, cfg):
    return nn.DataParallel(module) if cfg['multi_gpu'] else module


def build_detector(key, cfg, dim, probe=None):
    backbone = create_backbone(key)
    inject_lora(backbone, cfg['lora_targets'], cfg['lora_rank'], cfg['lora_alpha'], cfg['lora_dropout'])
    det = Detector(Encoder(backbone, cfg['amp']), dim)
    if probe is not None:
        with torch.no_grad():
            det.head.weight.copy_(torch.as_tensor(probe['w'], dtype=torch.float32)[None])
            det.head.bias.fill_(float(probe['b']))
    return det.to(cfg['device'])


def param_audit(det):
    total = sum(p.numel() for p in det.parameters())
    lora = sum(p.numel() for n, p in det.named_parameters() if '.lora.' in n)
    head = sum(p.numel() for p in det.head.parameters())
    trainable = sum(p.numel() for p in det.parameters() if p.requires_grad)
    if trainable != lora + head:
        raise RuntimeError('Something other than LoRA and the head is trainable')
    return dict(total_params=total, lora_params=lora, head_params=head,
                trainable_params=trainable, trainable_pct=100 * trainable / total)


# ------------------------------------------------------------- evaluation

def video_scores(frames, prob):
    d = frames[['dataset', 'video_id', 'group_id', 'label']].copy()
    d['probability'] = prob
    return d.groupby(['dataset', 'video_id', 'group_id', 'label'], as_index=False).probability.mean()


def macro_auc(v):
    return float(np.mean([roc_auc_score(g.label, g.probability) for _, g in v.groupby('dataset')]))


@torch.no_grad()
def predict(det, frames, tf, cfg):
    net = parallel(det, cfg).eval()
    prob = np.full(len(frames), np.nan)
    for x, _, idx in loader(frames, tf, cfg):
        prob[idx.numpy()] = torch.sigmoid(net(x.to(cfg['device'], non_blocking=True))).cpu().numpy()
    if not np.isfinite(prob).all():
        raise FloatingPointError('Non-finite predictions. Set CFG["amp"]=False (fp32, slower).')
    return video_scores(frames, prob)


def paired_delta(a, b, n_boot, seed):
    """AUC(b) - AUC(a) on identical videos; resample (dataset, group) clusters."""
    keys = ['dataset', 'video_id', 'group_id', 'label']
    d = a.merge(b, on=keys, validate='one_to_one', suffixes=('_a', '_b'))
    if not len(d) == len(a) == len(b):
        raise ValueError('The two arms did not score exactly the same videos')
    y, pa, pb = d.label.to_numpy(), d.probability_a.to_numpy(), d.probability_b.to_numpy()
    groups = list(d.groupby(['dataset', 'group_id']).indices.values())
    rng = np.random.default_rng(seed)
    samples = []
    for _ in range(n_boot):
        i = np.concatenate([groups[g] for g in rng.integers(0, len(groups), len(groups))])
        if len(np.unique(y[i])) == 2:
            samples.append(roc_auc_score(y[i], pb[i]) - roc_auc_score(y[i], pa[i]))
    if len(samples) < 0.9 * n_boot:
        raise ValueError('Too few valid bootstrap samples; inspect class/group balance')
    lo, hi = np.quantile(samples, [0.025, 0.975])
    return dict(delta=float(roc_auc_score(y, pb) - roc_auc_score(y, pa)),
                lo=float(lo), hi=float(hi))


# ----------------------------------------------------------- frozen probe

@torch.no_grad()
def extract_features(encoder, frames, tf, cfg, what):
    net = parallel(encoder, cfg).eval()
    out, t = None, time.time()
    dl = loader(frames, tf, cfg)
    for k, (x, _, idx) in enumerate(dl):
        f = net(x.to(cfg['device'], non_blocking=True)).cpu().numpy()
        if out is None:
            out = np.full((len(frames), f.shape[1]), np.nan, np.float32)
        out[idx.numpy()] = f
        if k % cfg['log_every'] == 0:
            log(f'  {what}: batch {k + 1}/{len(dl)}')
    if not np.isfinite(out).all():
        raise FloatingPointError(f'{what}: non-finite features. Set CFG["amp"]=False (fp32, slower).')
    log(f'  {what}: {len(frames):,} frames at {len(frames) / (time.time() - t):.0f} img/s')
    return out


def bank_features(key, bank, cfg, out, reuse_dirs):
    for d in [out, *map(Path, reuse_dirs)]:
        idx, feat = d / f'bank_{key}.csv', d / f'features_{key}.npy'
        if idx.is_file() and feat.is_file() and pd.read_csv(idx).path.tolist() == bank.path.tolist():
            log(f'Reusing cached {key} features from {d}')
            if d != out:
                shutil.copy(idx, out / idx.name)
                shutil.copy(feat, out / feat.name)
            return np.load(feat).astype(np.float32)
    backbone = create_backbone(key).to(cfg['device'])
    _, tf_eval, _ = backbone_transforms(backbone, cfg['img_size'])
    feats = extract_features(Encoder(backbone, cfg['amp']), bank, tf_eval, cfg, f'{key} feature bank')
    del backbone
    free()
    feats = feats.astype(np.float16)   # stored half precision; used identically fresh or reused
    np.save(out / f'features_{key}.npy', feats)
    bank[BANK_COLS].to_csv(out / f'bank_{key}.csv', index=False)
    return feats.astype(np.float32)


def fold_scaler(scaler, clf):
    """Logistic regression on standardised features -> one Linear on raw features."""
    w = clf.coef_[0] / scaler.scale_
    b = clf.intercept_[0] - np.sum(clf.coef_[0] * scaler.mean_ / scaler.scale_)
    return w.astype(np.float32), float(b)


def probe_prob(feats, probe):
    return expit(feats.astype(np.float64) @ probe['w'].astype(np.float64) + probe['b'])


def fit_probe(f_tr, y_tr, f_va, val, cfg):
    scaler = StandardScaler().fit(f_tr)
    x_tr, x_va = scaler.transform(f_tr), scaler.transform(f_va)
    grid, best = [], None
    for C in cfg['probe_C']:
        clf = LogisticRegression(C=C, max_iter=5000, class_weight='balanced').fit(x_tr, y_tr)
        auc = macro_auc(video_scores(val, clf.predict_proba(x_va)[:, 1]))
        grid.append(dict(C=C, val_auc=auc))
        if best is None or auc > best[1]:
            best = (clf, auc, C)
    w, b = fold_scaler(scaler, best[0])
    return dict(C=best[2], val_auc=best[1], w=w, b=b, grid=grid)


def run_probes(key, splits, folds, cfg, out, reuse_dirs):
    parts = [x for held in folds for x in splits[held]]
    bank = (pd.concat(parts).drop_duplicates('path').sort_values('path').reset_index(drop=True))
    log(f'Feature bank: {len(bank):,} frames from {bank.video_id.nunique():,} videos')
    feats = bank_features(key, bank, cfg, out, reuse_dirs)
    row_of = pd.Series(np.arange(len(bank)), index=bank.path)

    def get(frame):
        return feats[row_of.loc[frame.path].to_numpy()]

    probes, rows = {}, []
    for held in folds:
        train, val, test = splits[held]
        t = time.time()
        p = fit_probe(get(train), train.label.to_numpy(), get(val), val, cfg)
        p['val_pred'] = video_scores(val, probe_prob(get(val), p))
        p['test_pred'] = video_scores(test, probe_prob(get(test), p))
        p['test_auc'] = macro_auc(p['test_pred'])
        p['test_pred'].to_csv(out / f'pred_probe_{held}.csv', index=False)
        rows.append(dict(backbone=key, held_out=held, C=p['C'], val_auc=p['val_auc'],
                         test_auc=p['test_auc'], train_frames=len(train),
                         test_videos=len(p['test_pred']), fit_sec=round(time.time() - t, 1)))
        probes[held] = p
        log(f'  probe {held}: C={p["C"]:g} source-val AUC {p["val_auc"]:.4f} | '
            f'held-out AUC {p["test_auc"]:.4f}')
    pd.DataFrame(rows).to_csv(out / f'probe_{key}.csv', index=False)
    return probes, feats.shape[1]


# ----------------------------------------------------- memory and runtime

def _is_oom(e):
    return isinstance(e, torch.cuda.OutOfMemoryError) or 'out of memory' in str(e).lower()


def choose_memory_mode(det, batch, cfg, allow_ckpt):
    """Pick checkpointing and a micro-batch whose training step fits on GPU 0.

    Gradients are accumulated over micro-batches up to the full batch, so the
    optimiser update matches the other backbones. Smaller micro-batches are
    tried before checkpointing, which costs an extra forward pass.
    """
    x, y = batch
    total = torch.cuda.get_device_properties(0).total_memory
    sizes = [m for m in (len(x), len(x) // 2, len(x) // 4) if m >= 2]
    plans = [(False, m) for m in sizes] + ([(True, m) for m in sizes] if allow_ckpt else [])
    det.train()
    for ckpt, micro in plans:
        set_checkpointing(det.encoder.backbone, ckpt)
        det.zero_grad(set_to_none=True)
        free()
        torch.cuda.reset_peak_memory_stats(0)
        base = torch.cuda.memory_allocated(0)
        mode = (f'micro-batch {micro} of {len(x)}, checkpointing {"on" if ckpt else "off"}, '
                f'weights and state {base / 2**30:.1f} GiB')
        try:
            F.binary_cross_entropy_with_logits(parallel(det, cfg)(x[:micro]), y[:micro]).backward()
            peak = torch.cuda.max_memory_allocated(0)
            det.zero_grad(set_to_none=True)
            if peak < 0.85 * total:
                log(f'  memory: {mode}, peak {peak / 2**30:.1f} of {total / 2**30:.1f} GiB on GPU 0')
                return ckpt, micro, peak
            log(f'  memory: {mode}, peak {peak / 2**30:.1f} GiB is too close to the limit')
        except RuntimeError as e:
            if not _is_oom(e):
                raise
            log(f'  memory: {mode}, out of memory')
        det.zero_grad(set_to_none=True)
        free()
    raise RuntimeError(f'Even a micro-batch of {sizes[-1]} does not fit on GPU 0')


def benchmark(det, batches, val, tf_eval, cfg, micro):
    """Train speed excludes data loading (optimistic); eval speed includes it."""
    if len(batches) < 2:
        raise ValueError('Need at least two training batches to benchmark')
    net = parallel(det, cfg)
    det.train()
    per_img = []
    for i, (x, y) in enumerate(batches):
        torch.cuda.synchronize()
        t = time.time()
        for xc, yc in zip(x.split(micro), y.split(micro)):
            F.binary_cross_entropy_with_logits(net(xc), yc).backward()
        torch.cuda.synchronize()
        if i:
            per_img.append((time.time() - t) / len(x))
        det.zero_grad(set_to_none=True)
    n = min(len(val), cfg['eval_batch_size'] * 6)
    t = time.time()
    predict(det, val.head(n), tf_eval, cfg)
    return dict(train_img_s=float(1 / np.mean(per_img)), eval_img_s=float(n / (time.time() - t)))


def project_fold_hours(sizes, bench, cfg):
    n_train, n_val, n_test = sizes
    sec = (cfg['epochs'] * (n_train / bench['train_img_s'] + n_val / bench['eval_img_s'])
           + (n_val + n_test) / bench['eval_img_s'])
    return 1.15 * sec / 3600


# ---------------------------------------------------------------- testing

def unit_tests(key, cfg):
    """CPU only, random weights, no downloads. Uses the exact timm class at depth 2."""
    torch.manual_seed(0)
    rng = np.random.default_rng(0)
    x = torch.randn(5, 8)
    base = nn.Linear(8, 6)
    w = LoRALinear(base, 4, 8, 0.0)
    assert torch.allclose(w(x), base(x)), 'LoRALinear must start as the base layer'
    qkv = nn.Linear(8, 24)
    f = LoRAFusedQKV(qkv, 'qv', 4, 8, 0.0)
    with torch.no_grad():
        for a in f.lora.values():
            a.B.normal_()
        moved = (f(x) - qkv(x)).abs().reshape(5, 3, 8).amax(dim=(0, 2))
    assert moved[0] > 0 and moved[1] == 0 and moved[2] > 0, 'qv LoRA must leave k untouched'

    m = create_backbone(key, pretrained=False, depth=2).eval()
    xin = torch.randn(2, 3, 224, 224)
    with torch.no_grad():
        ref = m(xin)
    names = inject_lora(m, cfg['lora_targets'], 4, 8, 0.0)
    ads = adapters(m)
    with torch.no_grad():
        out = m(xin)
    assert len(names) == 2 and ads, f'{key}: LoRA not attached to both blocks'
    assert all(a.calls > 0 for a in ads), f'{key}: attention forward bypasses LoRA'
    assert torch.allclose(ref, out, atol=1e-5), f'{key}: zero-initialised LoRA changed the output'
    assert not [n for n, p in m.named_parameters() if p.requires_grad and '.lora.' not in n]
    head = nn.Linear(ref.shape[1], 1)
    for ckpt in (False, True):
        m.train()
        m.zero_grad(set_to_none=True)
        try:
            set_checkpointing(m, ckpt)
            head(m(xin)).sum().backward()
        except Exception as e:
            if not ckpt:
                raise
            CKPT_OK[key] = False
            set_checkpointing(m, False)
            print(f'WARNING: gradient checkpointing unavailable for {key} ({type(e).__name__}: {e}). '
                  f'Training will run without it; lower batch_size if memory runs out.')
            continue
        dead = [a for a in ads if a.B.grad is None or not a.B.grad.abs().sum() > 0]
        assert not dead, f'{key}: checkpointing={ckpt} leaves {len(dead)} adapters without gradient'
        if ckpt:
            CKPT_OK[key] = True

    X = rng.normal(size=(300, 12)) * rng.uniform(0.1, 5, 12) + rng.normal(size=12)
    y = (X[:, 0] + rng.normal(size=300) > X[:, 0].mean()).astype(int)
    sc = StandardScaler().fit(X)
    clf = LogisticRegression(max_iter=2000).fit(sc.transform(X), y)
    wv, bv = fold_scaler(sc, clf)
    assert np.allclose(expit(X @ wv.astype(np.float64) + bv),
                       clf.predict_proba(sc.transform(X))[:, 1], atol=1e-5), 'probe folding is wrong'

    rows = [dict(path=f'/x/{ds}/{v}/{k}.png', dataset=ds, video_id=f'{ds}::{v}',
                 group_id=f'{ds}::{v}', label=v % 2,
                 split='train' if v < 24 else ('val' if v < 32 else 'test'))
            for ds in CORPORA for v in range(40) for k in range(10)]
    toy = pd.DataFrame(rows)
    tcfg = dict(cfg, smoke=False, train_per_source=100, eval_frames_per_video=4)
    for held in CORPORA:
        tr, va, te = fold_frames(toy, held, tcfg)
        assert held not in set(tr.dataset) | set(va.dataset)
        assert te.video_id.nunique() == 40 and len(te) == 160
        assert (tr.groupby('dataset').size() == 100).all() and (tr.label.value_counts() == 100).all()
        assert tr.groupby('dataset').video_id.nunique().min() == 24
    v = video_scores(te, rng.uniform(size=len(te)))
    r = paired_delta(v, v, 50, 0)
    assert r['delta'] == 0 and r['lo'] == r['hi'] == 0
    print(f'PASS: unit tests ({key} architecture at depth 2, LoRA wiring, zero init, '
          f'gradients, probe folding, fold leakage, bootstrap)')


def preflight(key, cfg, splits, lora_folds, probe_folds, out):
    """Real pretrained weights: LoRA wiring, zero-init identity, gradients, memory, speed."""
    dev = cfg['device']
    first = splits[lora_folds[0] if lora_folds else probe_folds[0]]
    backbone = create_backbone(key).to(dev).eval()
    tf_train, tf_eval, norm = backbone_transforms(backbone, cfg['img_size'])
    x = torch.stack([tf_eval(Image.open(p).convert('RGB')) for p in first[0].path.head(4)]).to(dev)
    enc32 = Encoder(backbone, amp=False)
    with torch.no_grad():
        before = enc32(x)
    inject_lora(backbone, cfg['lora_targets'], cfg['lora_rank'], cfg['lora_alpha'], cfg['lora_dropout'])
    backbone.to(dev)   # the new LoRA weights are created on the CPU
    ads = adapters(backbone)
    with torch.no_grad():
        after = enc32(x)
        assert all(a.calls > 0 for a in ads), 'Some LoRA adapters never ran'
        assert torch.allclose(before, after, atol=1e-4, rtol=1e-4), 'Zero-initialised LoRA changed features'
        for a in ads:
            a.B.normal_(0, 0.02)
        moved = (enc32(x) - before).abs().max().item()
        for a in ads:
            a.B.zero_()
    assert moved > 1e-4, 'Perturbing LoRA did not change the features'

    dim = before.shape[1]
    det = Detector(Encoder(backbone, amp=False), dim).to(dev)
    with torch.no_grad():
        det.head.weight.normal_(0, 0.05)
    for ckpt in ((False, True) if CKPT_OK.get(key) else (False,)):
        set_checkpointing(backbone, ckpt)
        det.train()
        det.zero_grad(set_to_none=True)
        F.binary_cross_entropy_with_logits(det(x[:2]), torch.tensor([0., 1.], device=dev)).backward()
        dead = [a for a in ads if a.B.grad is None or not a.B.grad.abs().sum() > 0]
        assert not dead, f'checkpointing={ckpt}: {len(dead)} adapters got no gradient'
        assert all(torch.isfinite(p.grad).all() for p in det.parameters() if p.grad is not None)
    audit = param_audit(det)
    det.encoder.amp = cfg['amp']
    det.zero_grad(set_to_none=True)
    log(f'PASS: real {key}: {len(ads)} adapters on {len(backbone.blocks)} blocks all run, '
        f'zero init leaves features unchanged, gradients reach every adapter '
        f'(frozen weights {det.encoder.in_dtype})')
    log(f'  trainable {audit["trainable_params"]:,} of {audit["total_params"]:,} '
        f'({audit["trainable_pct"]:.2f}%), feature dim {dim}, '
        f'pooling {getattr(backbone, "global_pool", "?")!r}, normalisation {norm}')

    batches = [(xb.to(dev), yb.float().to(dev)) for xb, yb, _ in
               itertools.islice(loader(first[0], tf_train, cfg, train=True), 4)]
    ckpt, micro, peak = choose_memory_mode(det, batches[0], cfg, allow_ckpt=CKPT_OK.get(key, False))
    bench = benchmark(det, batches, first[1], tf_eval, cfg, micro)
    pooling = getattr(backbone, 'global_pool', '?')
    del det, backbone, batches, x
    free()

    n_bank = len(pd.concat([p for h in probe_folds for p in splits[h]]).drop_duplicates('path'))
    plan = [dict(step='feature bank (frozen probe)', frames=n_bank, hours=1.15 * n_bank / bench['eval_img_s'] / 3600)]
    for held in lora_folds:
        tr, va, te = splits[held]
        plan.append(dict(step=f'LoRA, {held} held out', frames=len(tr),
                         hours=project_fold_hours((len(tr), len(va), len(te)), bench, cfg)))
    plan = pd.DataFrame(plan)
    log(f'  speed: train {bench["train_img_s"]:.0f} img/s, eval {bench["eval_img_s"]:.0f} img/s '
        f'on {torch.cuda.device_count()} GPU(s)')
    print(plan.round(2).to_string(index=False))
    total = hours() + plan.hours.sum()
    print(f'Projected finish at {total:.1f} h of this session (limit {cfg["time_limit_h"]} h). '
          f'Folds that would cross the limit are skipped and can run in a later session.')
    result = dict(dim=dim, audit=audit, bench=bench, grad_ckpt=ckpt, micro_batch=micro, peak_gib=peak / 2**30,
                  normalisation=norm, pooling=str(pooling), plan=plan.to_dict('records'))
    (out / 'preflight.json').write_text(json.dumps(jsonable(result), indent=2))
    return result


# --------------------------------------------------------------- training

def signature(key, cfg):
    keys = ['seed', 'smoke', 'train_per_source', 'eval_frames_per_video', 'lora_rank', 'lora_alpha',
            'lora_dropout', 'lora_targets', 'epochs', 'batch_size', 'lr_lora', 'lr_head', 'weight_decay']
    return dict(backbone=key, data_seed=cfg.get('data_seed', cfg['seed']), **{k: cfg[k] for k in keys})


def reuse_fold(key, held, cfg, out, reuse_dirs):
    for d in map(Path, reuse_dirs):
        meta = d / f'lora_{held}.json'
        if meta.is_file():
            r = json.loads(meta.read_text())
            if r.get('status') == 'complete' and r.get('signature') == jsonable(signature(key, cfg)):
                for name in [meta.name, f'pred_lora_{held}.csv', f'history_{held}.csv', f'lora_{held}_best.pt']:
                    if (d / name).is_file():
                        shutil.copy(d / name, out / name)
                r['test_pred'] = pd.read_csv(out / f'pred_lora_{held}.csv', dtype={'video_id': str, 'group_id': str})
                log(f'Reusing finished LoRA fold {held} from {d}')
                return r
    return None


def run_lora_fold(key, held, split, probe, dim, cfg, out, bench, reuse_dirs):
    done = reuse_fold(key, held, cfg, out, reuse_dirs)
    if done:
        return done
    train, val, test = split
    projected = project_fold_hours((len(train), len(val), len(test)), bench, cfg)
    if hours() + projected > cfg['time_limit_h']:
        log(f'SKIP LoRA {held}: needs ~{projected:.1f} h, {cfg["time_limit_h"] - hours():.1f} h left. '
            f'Run it in a new session with this output in REUSE_DIRS.')
        return dict(status='skipped_time_budget', projected_h=projected)
    log(f'=== LoRA {key} | {held} held out | {len(train):,} train frames | ~{projected:.1f} h ===')
    seed_all(cfg['seed'])
    dev = cfg['device']
    det = build_detector(key, cfg, dim, probe)
    audit = param_audit(det)
    tf_train, tf_eval, _ = backbone_transforms(det.encoder.backbone, cfg['img_size'])
    train_dl = loader(train, tf_train, cfg, train=True)
    xb, yb, _ = next(iter(train_dl))
    ckpt, micro, _ = choose_memory_mode(det, (xb.to(dev), yb.float().to(dev)), cfg,
                                        allow_ckpt=CKPT_OK.get(key, False))
    del xb, yb
    with torch.no_grad():   # choose_memory_mode ran backward only; make sure nothing moved
        assert all(not a.B.abs().sum() for a in adapters(det))

    net = parallel(det, cfg)
    trainable = [p for p in det.parameters() if p.requires_grad]
    lora_p = [p for n, p in det.named_parameters() if p.requires_grad and not n.startswith('head.')]
    opt = torch.optim.AdamW([dict(params=lora_p, lr=cfg['lr_lora'], weight_decay=cfg['weight_decay']),
                             dict(params=list(det.head.parameters()), lr=cfg['lr_head'], weight_decay=0.0)])
    total = cfg['epochs'] * len(train_dl)
    warm = max(1, int(cfg['warmup_frac'] * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm
        else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler('cuda', enabled=cfg['amp'])
    n_fake = int((train.label == 1).sum())
    pos_weight = torch.tensor(int((train.label == 0).sum()) / max(1, n_fake), device=dev)

    # Epoch 0 is the probe pushed through the network: a check on LoRA zero init and head folding.
    v0 = predict(det, val, tf_eval, cfg)
    gap = v0.merge(probe['val_pred'], on=['dataset', 'video_id', 'group_id', 'label'])
    gap = (gap.probability_x - gap.probability_y).abs()
    log(f'  epoch 0 vs probe on source val: mean |dp| {gap.mean():.5f}, max {gap.max():.4f}')
    if gap.mean() > 0.01 or gap.max() > 0.1:
        raise AssertionError('Epoch-0 LoRA model does not reproduce the probe; do not trust this fold')
    history = [dict(epoch=0, train_loss=np.nan, val_auc=macro_auc(v0), sec=0.0)]
    best_auc, best_epoch, best_state, best_val = -1.0, None, None, None
    status, t_fold = 'complete', time.time()
    for epoch in range(1, cfg['epochs'] + 1):
        det.train()
        t, losses = time.time(), []
        for step, (x, y, _) in enumerate(train_dl):
            x, y = x.to(dev, non_blocking=True), y.float().to(dev, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            batch_loss = 0.0
            for xc, yc in zip(x.split(micro), y.split(micro)):   # one chunk unless memory needs more
                loss = F.binary_cross_entropy_with_logits(net(xc), yc, pos_weight=pos_weight) * (len(xc) / len(x))
                if not torch.isfinite(loss):
                    raise FloatingPointError(f'Non-finite loss at epoch {epoch} step {step}. '
                                             f'Try CFG["amp"]=False or a lower lr_lora.')
                scaler.scale(loss).backward()
                batch_loss += loss.item()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable, cfg['grad_clip'])
            scaler.step(opt)
            scaler.update()
            sched.step()
            losses.append(batch_loss)
            if step % cfg['log_every'] == 0:
                log(f'  epoch {epoch} step {step + 1}/{len(train_dl)} loss {np.mean(losses[-50:]):.4f}')
        vp = predict(det, val, tf_eval, cfg)
        auc = macro_auc(vp)
        history.append(dict(epoch=epoch, train_loss=float(np.mean(losses)), val_auc=auc,
                            sec=round(time.time() - t, 1)))
        pd.DataFrame(history).to_csv(out / f'history_{held}.csv', index=False)
        log(f'  epoch {epoch}: train loss {np.mean(losses):.4f} | source-val AUC {auc:.4f}')
        if auc > best_auc:
            best_auc, best_epoch, best_val = auc, epoch, vp
            best_state = {n: p.detach().cpu().clone() for n, p in det.named_parameters() if p.requires_grad}
            torch.save(best_state, out / f'lora_{held}_best.pt')
        if hours() > cfg['time_limit_h'] and epoch < cfg['epochs']:
            status = 'truncated'
            log(f'  time limit reached after epoch {epoch}; scoring the best epoch so far')
            break
    _, unexpected = det.load_state_dict(best_state, strict=False)
    assert not unexpected
    test_pred = predict(det, test, tf_eval, cfg)
    test_pred.to_csv(out / f'pred_lora_{held}.csv', index=False)
    best_val.to_csv(out / f'pred_lora_val_{held}.csv', index=False)
    result = dict(status=status, signature=signature(key, cfg), held_out=held, best_epoch=best_epoch,
                  val_auc=best_auc, test_auc=macro_auc(test_pred), history=history, audit=audit,
                  grad_ckpt=ckpt, micro_batch=micro, train_hours=(time.time() - t_fold) / 3600)
    (out / f'lora_{held}.json').write_text(json.dumps(jsonable(result), indent=2))
    log(f'  LoRA {held}: best epoch {best_epoch} (source-val {best_auc:.4f}) | '
        f'held-out AUC {result["test_auc"]:.4f} | probe {probe["test_auc"]:.4f}')
    del det, net, opt, best_state
    free()
    result['test_pred'] = test_pred
    return result


# -------------------------------------------------------------- reference

def reference_predictions(held, test, cfg, out):
    """Phase 11 dual-stream and baseline checkpoints re-scored on these exact frames.
    Needs Cell 1 (create_model, load_checkpoint) and Cell 2 (DualNet, eval_tf)."""
    refs = {}
    for arm, fname in [('dualstream', f'model_dualstream_loo_{held}.pth'),
                       ('baseline', f'model_baseline_loo_{held}.pth')]:
        hits = [p for p in cfg['checkpoints'] if p.name == fname]
        if not hits:
            log(f'  reference {arm}/{held}: {fname} not attached, skipped')
            continue
        try:
            if arm == 'dualstream':
                model = DualNet(pretrained=False)
                model.load_state_dict(torch.load(hits[0], map_location='cpu', weights_only=True), strict=True)
            else:
                model = load_checkpoint(create_model(pretrained=False), hits[0])
        except Exception as e:
            log(f'  reference {arm}/{held}: could not load {hits[0]} ({e}); skipped')
            continue
        model = model.to(cfg['device']).eval()
        prob = np.full(len(test), np.nan)
        with torch.no_grad():
            for x, _, idx in loader(test, eval_tf, cfg):
                with torch.autocast('cuda', dtype=torch.float16, enabled=cfg['amp']):
                    logit = model(x.to(cfg['device'], non_blocking=True)).float().reshape(-1)
                prob[idx.numpy()] = torch.sigmoid(logit).cpu().numpy()
        if not np.isfinite(prob).all():
            log(f'  reference {arm}/{held}: non-finite outputs, skipped')
            continue
        refs[arm] = video_scores(test, prob)
        refs[arm].to_csv(out / f'pred_{arm}_{held}.csv', index=False)
        log(f'  reference {arm}/{held}: AUC {macro_auc(refs[arm]):.4f} on the same frames ({hits[0]})')
        del model
        free()
    return refs


# ------------------------------------------------------- earlier outputs

def find_input_dirs(filename, skip=()):
    """Folders under /kaggle/input holding `filename`, without listing the face-crop trees."""
    skip, hits = {str(p) for p in skip}, []
    for dp, dns, fns in os.walk('/kaggle/input'):
        if filename in fns:
            hits.append(Path(dp))
        if dp in skip or dp.count(os.sep) >= 9 or any(
                f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')) for f in fns):
            dns[:] = []
    return sorted(hits)


def crop_size_report(d, n, seed):
    """Face-crop sizes from a sample per corpus: decides whether DINOv2 above 224 px is worth testing."""
    rows = []
    for ds, g in d.groupby('dataset'):
        for path in g.path.sample(min(n, len(g)), random_state=seed):
            with Image.open(path) as im:
                rows.append(dict(dataset=ds, short_side=min(im.size), long_side=max(im.size)))
    s = pd.DataFrame(rows)
    return s.groupby('dataset').agg(
        sampled=('short_side', 'size'), short_side_p10=('short_side', lambda v: v.quantile(0.1)),
        short_side_median=('short_side', 'median'), short_side_p90=('short_side', lambda v: v.quantile(0.9)),
        share_at_least_448=('short_side', lambda v: float((v >= 448).mean()))).reset_index()


# ---------------------------------------------------------------- outputs

def summarise(key, probes, lora, refs, cfg, out):
    rows = []
    for held, p in probes.items():
        row = dict(backbone=key, held_out=held, test_videos=len(p['test_pred']),
                   probe_val_auc=p['val_auc'], probe_auc=p['test_auc'])
        r = lora.get(held)
        ok = bool(r) and r['status'] in ('complete', 'truncated')
        if ok:
            c = paired_delta(p['test_pred'], r['test_pred'], cfg['bootstrap'], cfg['seed'])
            row.update(lora_val_auc=r['val_auc'], lora_auc=r['test_auc'], lora_epoch=r['best_epoch'],
                       lora_minus_probe=c['delta'], lora_minus_probe_lo=c['lo'],
                       lora_minus_probe_hi=c['hi'])
        row['lora_status'] = r['status'] if r else 'not_run'
        for arm, pred in refs.get(held, {}).items():
            row[f'{arm}_auc'] = macro_auc(pred)
            if ok:
                c = paired_delta(pred, r['test_pred'], cfg['bootstrap'], cfg['seed'])
                row.update({f'lora_minus_{arm}': c['delta'], f'lora_minus_{arm}_lo': c['lo'],
                            f'lora_minus_{arm}_hi': c['hi']})
        rows.append(row)
    table = pd.DataFrame(rows)
    table.to_csv(out / f'faculty_table_{key}.csv', index=False)
    return table


def make_plots(key, table, lora, out):
    import matplotlib.pyplot as plt
    arms = [c for c in ['probe_auc', 'lora_auc', 'dualstream_auc', 'baseline_auc'] if c in table]
    labels = {'probe_auc': 'frozen probe', 'lora_auc': 'LoRA',
              'dualstream_auc': 'dual-stream (Phase 11)', 'baseline_auc': 'baseline (Phase 11)'}
    fig, ax = plt.subplots(figsize=(7, 4))
    width = 0.8 / len(arms)
    for i, arm in enumerate(arms):
        ax.bar(np.arange(len(table)) + i * width, table[arm], width, label=labels[arm])
    ax.axhline(0.5, color='grey', lw=0.8, ls='--')
    ax.set_xticks(np.arange(len(table)) + 0.4 - width / 2)
    ax.set_xticklabels([f'{h} held out' for h in table.held_out])
    ax.set_ylim(0.4, 1.0)
    ax.set_ylabel('video-level AUC (same frames)')
    ax.set_title(f'{BACKBONES[key]["timm_id"]}: unseen-dataset AUC')
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(out / f'heldout_auc_{key}.png', dpi=160)
    plt.close(fig)

    runs = {h: r for h, r in lora.items() if r.get('history')}
    if runs:
        fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
        for held, r in runs.items():
            h = pd.DataFrame(r['history'])
            axes[0].plot(h.epoch, h.val_auc, marker='o', label=f'{held} held out')
            axes[1].plot(h.epoch[1:], h.train_loss[1:], marker='o', label=f'{held} held out')
        axes[0].set(xlabel='epoch (0 = frozen probe)', ylabel='source-val AUC', title='Model selection')
        axes[1].set(xlabel='epoch', ylabel='train BCE', title='Training loss')
        axes[0].legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(out / f'lora_curves_{key}.png', dpi=160)
        plt.close(fig)


def write_metadata(key, cfg, pre, lora, split_origin, out):
    meta = dict(
        backbone=key, timm_id=BACKBONES[key]['timm_id'], note=BACKBONES[key]['note'],
        torch=torch.__version__, timm=timm.__version__,
        gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
        wall_hours=hours(), split_origin=split_origin,
        config={k: v for k, v in cfg.items() if k != 'checkpoints'},
        preflight=pre, lora={h: {k: v for k, v in r.items() if k != 'test_pred'} for h, r in lora.items()})
    (out / 'run_metadata.json').write_text(json.dumps(jsonable(meta), indent=2))


def zip_results(out):
    """Everything except the cached feature bank (kept in /kaggle/working for reuse)."""
    faulthandler.cancel_dump_traceback_later()   # the run is finished: disarm the hang watchdog
    archive = out.with_suffix('.zip')
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in sorted(out.rglob('*')):
            if p.is_file() and p.suffix != '.npy':
                z.write(p, p.relative_to(out.parent))
    return archive


## Cell 4: codec7 module (H.264 round trip, scoring, paired bootstrap, mask tables), the same file as Phase 14b and Gate 0

In [ ]:
%%writefile codec7.py
"""Phase 14b: codec-aware block-DCT masking, the full three-fold experiment.

Every arm continues the Phase 11 checkpoint of one leave-one-dataset-out fold on
the Phase 13 frames with the same schedule, then scores the held-out corpus clean,
after JPEG re-saving and after H.264 re-encoding. Masks act only on the input of
the dual-stream model's `fad` module and only in training; the RGB stream never
sees them. Checkpoints are chosen on source validation only.

Self-contained: model builders, the transform, the log function and the session
clock come in as arguments. Nothing trains on import.
"""
import faulthandler
import gc
import hashlib
import io
import json
import math
import multiprocessing
import os
import random
import shutil
import subprocess
import tempfile
import time
import traceback
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score


CORPORA = ('celebdf', 'ffpp', 'dfdc')
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
JPEG_LUMA = [
    [16, 11, 10, 16, 24, 40, 51, 61], [12, 12, 14, 19, 26, 58, 60, 55],
    [14, 13, 16, 24, 40, 57, 69, 56], [14, 17, 22, 29, 51, 87, 80, 62],
    [18, 22, 37, 56, 68, 109, 103, 77], [24, 35, 55, 64, 81, 104, 113, 92],
    [49, 64, 78, 87, 103, 121, 120, 101], [72, 92, 95, 98, 112, 100, 103, 99],
]
MASK_KINDS = ('qtable', 'uniform', 'inverted')
ARM_DEFS = {
    'control':        dict(model='dual', mask=None,     realjpeg=False, trained=True),
    'qmask':          dict(model='dual', mask='qtable', realjpeg=False, trained=True),
    'randmask':       dict(model='dual', mask='uniform', realjpeg=False, trained=True),
    'invmask':        dict(model='dual', mask='inverted', realjpeg=False, trained=True),
    'realjpeg':       dict(model='dual', mask=None,     realjpeg=True,  trained=True),
    'realjpeg_qmask': dict(model='dual', mask='qtable', realjpeg=True,  trained=True),
    'rgb_control':    dict(model='rgb',  mask=None,     realjpeg=False, trained=True),
    'rgb_realjpeg':   dict(model='rgb',  mask=None,     realjpeg=True,  trained=True),
}
ORIGINAL_DEFS = {
    'dual_original': dict(model='dual', mask=None, realjpeg=False, trained=False),
    'rgb_original':  dict(model='rgb',  mask=None, realjpeg=False, trained=False),
}
ARM_ORDER = list(ORIGINAL_DEFS) + list(ARM_DEFS)
# Phase 13b Part A: the untouched checkpoints on exactly these test frames (no flip TTA).
REFERENCE_AUC = {'dual_original': dict(celebdf=0.8081, ffpp=0.8719, dfdc=0.7402),
                 'rgb_original':  dict(celebdf=0.7759, ffpp=0.8320, dfdc=0.7414)}
COMPARISONS = [('qmask', 'control'), ('qmask', 'randmask'), ('qmask', 'invmask'), ('qmask', 'realjpeg'),
               ('realjpeg_qmask', 'realjpeg'), ('randmask', 'control'), ('realjpeg', 'control'),
               ('control', 'rgb_control'), ('rgb_realjpeg', 'rgb_control')]
ENDPOINTS = {('qmask', 'randmask', 'jpeg20'): 'primary', ('qmask', 'control', 'jpeg20'): 'primary',
             ('qmask', 'randmask', 'h264_crf40'): 'secondary', ('qmask', 'control', 'h264_crf40'): 'secondary'}
SIGNATURE_KEYS = ('seed', 'data_seed', 'smoke', 'smoke_videos', 'train_per_source', 'eval_frames_per_video',
                  'epochs', 'lr', 'weight_decay', 'batch_size', 'grad_clip', 'amp', 'deterministic',
                  'max_drop', 'mask_seed_offset', 'realjpeg_p', 'realjpeg_q', 'realjpeg_seed_offset',
                  'jpeg_subsampling', 'eval_jpeg')
H264_KEYS = ('h264_crf', 'h264_videos_per_class', 'h264_seed', 'h264_preset', 'h264_fps', 'h264_size')
# Used only when Cell 6's benchmark has not run; the preflight replaces them with measured speeds.
DEFAULT_BENCH = {'dual': dict(train_img_s=120.0, eval_img_s=350.0), 'rgb': dict(train_img_s=250.0, eval_img_s=500.0)}
H264_SEC_PER_VIDEO = 0.6   # projection only: read 8 crops, encode and decode one clip, one process
WARMUP_SKIP_LIMIT = 20     # 20 halvings take GradScaler from 2**16 to 2**-4: beyond that it is not warm-up


def arm_spec(name):
    if name in ARM_DEFS:
        return ARM_DEFS[name]
    if name in ORIGINAL_DEFS:
        return ORIGINAL_DEFS[name]
    raise KeyError(f'Unknown arm {name!r}; choose from {ARM_ORDER}')


def seed_all(seed, deterministic=True):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = not deterministic
    torch.backends.cudnn.deterministic = deterministic


def free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def jsonable(x):
    if isinstance(x, dict):
        return {str(k): jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [jsonable(v) for v in x]
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, np.floating):
        return float(x)
    if isinstance(x, np.bool_):
        return bool(x)
    if isinstance(x, Path):
        return str(x)
    return x


def write_json(path, obj):
    Path(path).write_text(json.dumps(jsonable(obj), indent=2))


def file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()


def conditions(cfg, h264=True):
    names = ['clean'] + [f'jpeg{q}' for q in cfg['eval_jpeg']]
    if h264:
        names += ['clean_h264subset'] + [f'h264_crf{c}' for c in cfg['h264_crf']]
    return names


def read_pred(path):
    return pd.read_csv(path, dtype={'dataset': str, 'video_id': str, 'group_id': str})


# ------------------------------------------------------------------ masks

def drop_table(kind, max_drop=0.35):
    """8x8 drop probabilities. All three kinds have the same mean over the 63 AC
    coefficients; DC is never dropped."""
    q = np.array(JPEG_LUMA, dtype=np.float64)
    ac = np.ones((8, 8), bool)
    ac[0, 0] = False
    table = max_drop * q / q.max()
    table[0, 0] = 0.0
    target = table[ac].mean()
    if kind == 'qtable':
        p = table
    elif kind == 'uniform':
        p = np.where(ac, target, 0.0)
    elif kind == 'inverted':
        # Same probabilities, reversed order: the least-quantized coefficient gets the
        # largest probability. Tied table entries share their mean, which keeps the sum.
        qa, pa = q[ac], table[ac]
        inv = np.empty_like(pa)
        inv[np.argsort(qa, kind='stable')] = np.sort(pa)[::-1]
        _, tie = np.unique(qa, return_inverse=True)
        inv = (np.bincount(tie, inv) / np.bincount(tie))[tie]
        inv = np.clip(inv * target / inv.mean(), 0.0, 1.0)
        p = np.zeros((8, 8))
        p[ac] = inv
    else:
        raise ValueError(f'Unknown mask kind {kind!r}; choose from {MASK_KINDS}')
    if abs(p[ac].mean() - target) > 1e-6 or p[0, 0] != 0 or p.min() < 0 or p.max() > 1:
        raise AssertionError(f'{kind}: mean AC drop {p[ac].mean():.8f} != {target:.8f}')
    return p


class BlockDCTMask(nn.Module):
    """One stochastic 8x8 coefficient mask per image, shared by its channels and blocks.

    No inverted-dropout rescaling: removing AC energy is the intervention. The mask
    draws from its own generator, so it never advances the global RNG.
    """
    def __init__(self, probability, seed=1042):
        super().__init__()
        p = torch.as_tensor(np.asarray(probability), dtype=torch.float32)
        if p.shape != (8, 8) or p[0, 0] != 0 or p.min() < 0 or p.max() > 1:
            raise ValueError('probability must be an 8x8 table in [0, 1] with DC = 0')
        n = torch.arange(8, dtype=torch.float32)
        basis = torch.cos(math.pi * (2 * n + 1) * n[:, None] / 16) * math.sqrt(2 / 8)
        basis[0] /= math.sqrt(2)
        self.register_buffer('basis', basis)
        self.register_buffer('probability', p)
        self.active = bool((p > 0).any())
        self.seed = seed
        self.generators = {}
        self.calls = 0

    def coefficients(self, x):
        b, c, h, w = x.shape
        if h % 8 or w % 8:
            raise ValueError('FAD input H/W must be divisible by 8')
        blocks = x.float().reshape(b, c, h // 8, 8, w // 8, 8).permute(0, 1, 2, 4, 3, 5)
        return self.basis @ blocks @ self.basis.T

    def reconstruct(self, coeff):
        b, c, nh, nw, _, _ = coeff.shape
        blocks = self.basis.T @ coeff @ self.basis
        return blocks.permute(0, 1, 2, 4, 3, 5).reshape(b, c, nh * 8, nw * 8)

    def keep_mask(self, n, device):
        key = str(device)
        if key not in self.generators:
            self.generators[key] = torch.Generator(device=device).manual_seed(self.seed)
        u = torch.rand((n, 1, 1, 1, 8, 8), device=device, generator=self.generators[key])
        return u >= self.probability.to(device)

    def forward(self, x):
        if not self.training or not self.active:
            return x
        if x.ndim != 4 or not x.is_floating_point():
            raise ValueError('Expected floating BCHW spatial input to fad')
        self.calls += 1
        with torch.autocast(device_type=x.device.type, enabled=False):   # DCT stays fp32
            coeff = self.coefficients(x)
            out = self.reconstruct(coeff * self.keep_mask(len(x), x.device).to(coeff.dtype))
        return out.to(x.dtype)


class MaskedFrequencyModel(nn.Module):
    """Masks the input of base.fad through a forward pre-hook; base.rgb is untouched."""
    def __init__(self, base, mask):
        super().__init__()
        if not hasattr(base, 'fad') or not hasattr(base, 'rgb'):
            raise ValueError('Use the dual-stream model with .fad and .rgb modules')
        self.base, self.mask = base, mask
        self.handle = base.fad.register_forward_pre_hook(self._mask_input)

    def _mask_input(self, module, args):
        if not (module.training and self.mask.training):   # off whenever base or wrapper is in eval
            return None
        if len(args) != 1:
            raise ValueError('Expected fad(x); adapt the hook for another signature')
        return (self.mask(args[0]),)

    def remove(self):
        self.handle.remove()

    def forward(self, x):
        return self.base(x)


# ------------------------------------------------------------------- data

def jpeg_roundtrip(image, quality, subsampling=2):
    buf = io.BytesIO()
    image.save(buf, format='JPEG', quality=int(quality), subsampling=subsampling)
    buf.seek(0)
    with Image.open(buf) as decoded:
        return decoded.convert('RGB')


def realjpeg_params(seed, epoch, index, p, q_range):
    """Deterministic per (seed, epoch, sample), so it is identical in any DataLoader worker."""
    rng = np.random.default_rng([int(seed), int(epoch), int(index)])
    apply = bool(rng.random() < p)
    return apply, int(rng.integers(q_range[0], q_range[1] + 1))


class H264Store:
    """Decoded H.264 frames (memory-mapped uint8 array) looked up by original frame path."""
    def __init__(self, array, paths):
        self.array = array
        self.row = {p: i for i, p in enumerate(paths)}

    def __getitem__(self, path):
        return np.ascontiguousarray(self.array[self.row[path]])


class FrameData(Dataset):
    def __init__(self, frame, preprocess, quality=None, realjpeg=None, seed=0, store=None, subsampling=2):
        self.paths = frame.path.tolist()
        self.labels = frame.label.astype(float).tolist()
        self.preprocess, self.quality, self.store = preprocess, quality, store
        self.realjpeg, self.seed, self.subsampling = realjpeg, seed, subsampling
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = epoch   # workers are re-forked each epoch, so they see the new value

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        if self.store is not None:
            image = Image.fromarray(self.store[self.paths[i]])
        else:
            image = _open_rgb(self.paths[i])
        if self.quality is not None:
            image = jpeg_roundtrip(image, self.quality, self.subsampling)
        if self.realjpeg is not None:
            apply, q = realjpeg_params(self.seed, self.epoch, i, *self.realjpeg)
            if apply:
                image = jpeg_roundtrip(image, q, self.subsampling)
        return self.preprocess(image), self.labels[i], i


def _open_rgb(path):
    with Image.open(path) as im:
        return im.convert('RGB')


def loader(ds, cfg, train=False):
    return DataLoader(ds, batch_size=cfg['batch_size'] if train else cfg['eval_batch_size'],
                      shuffle=train, drop_last=train, num_workers=cfg['workers'],
                      timeout=600 if cfg['workers'] else 0,   # a stuck worker raises instead of hanging
                      pin_memory=cfg['device'].startswith('cuda'),
                      generator=torch.Generator().manual_seed(cfg['seed']))


def video_scores(frames, prob):
    d = frames[['dataset', 'video_id', 'group_id', 'label']].copy()
    d['probability'] = prob
    return d.groupby(['dataset', 'video_id', 'group_id', 'label'], as_index=False).probability.mean()


def macro_auc(v):
    return float(np.mean([roc_auc_score(g.label, g.probability) for _, g in v.groupby('dataset')]))


@torch.no_grad()
def predict(model, frames, preprocess, cfg, quality=None, store=None, log=None, what=''):
    """Video score = mean frame probability, fp16 autocast, no flip TTA (as Phase 13b Part A)."""
    model.eval()
    dev = cfg['device']
    prob = np.full(len(frames), np.nan)
    dl = loader(FrameData(frames, preprocess, quality=quality, store=store,
                          subsampling=cfg['jpeg_subsampling']), cfg)
    for k, (x, _, idx) in enumerate(dl):
        with torch.autocast(dev.split(':')[0], dtype=torch.float16, enabled=cfg['amp']):
            logit = model(x.to(dev, non_blocking=True)).float().reshape(-1)
        prob[idx.numpy()] = torch.sigmoid(logit).cpu().numpy()
        if log and k and k % cfg['log_every'] == 0:
            log(f'    {what}: batch {k + 1}/{len(dl)}')
    if not np.isfinite(prob).all():
        raise FloatingPointError(f'{what}: non-finite predictions')
    return video_scores(frames, prob)


# --------------------------------------------------------------- training

def train_step(model, x, y, criterion, opt, scaler, cfg):
    """One AMP step. Returns (loss, overflow). An fp16 overflow is not an error:
    GradScaler skips the step and lowers its scale, so the norm is only counted."""
    opt.zero_grad(set_to_none=True)
    with torch.autocast(cfg['device'].split(':')[0], enabled=cfg['amp']):
        loss = criterion(model(x).reshape(-1).float(), y)
    if not torch.isfinite(loss):
        raise FloatingPointError('non-finite training loss')
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    norm = nn.utils.clip_grad_norm_([p for p in model.parameters() if p.grad is not None], cfg['grad_clip'])
    scaler.step(opt)
    scaler.update()
    return float(loss.detach()), not bool(torch.isfinite(norm))


def check_overflow(skipped, steps, what, frac=0.05, floor=3):
    # The floor: on a short epoch (smoke, ~12 steps) one re-descent of the scale on a
    # batch with larger gradients is normal fp16 behaviour, not divergence.
    if skipped > max(frac * steps, floor):
        raise FloatingPointError(f'{what}: {skipped} of {steps} steps overflowed after GradScaler warm-up; '
                                 f'this is not warm-up. Try amp=False or a lower lr.')


# ------------------------------------------------------------------ H.264

def h264_roundtrip(frames, crf, exe, preset='medium', fps=25, threads=1, timeout=120):
    """uint8 (n, h, w, 3) RGB -> one libx264 clip -> decoded uint8 (n, h, w, 3). A hung ffmpeg
    raises TimeoutExpired instead of blocking the Pool (and the watchdog) forever."""
    frames = np.ascontiguousarray(frames, dtype=np.uint8)
    n, h, w, _ = frames.shape
    with tempfile.TemporaryDirectory(prefix='codec7_') as tmp:
        clip = os.path.join(tmp, 'clip.mp4')
        enc = [exe, '-hide_banner', '-loglevel', 'error', '-y',
               '-f', 'rawvideo', '-pix_fmt', 'rgb24', '-s', f'{w}x{h}', '-r', str(fps), '-i', 'pipe:0',
               '-c:v', 'libx264', '-preset', preset, '-crf', str(crf), '-pix_fmt', 'yuv420p',
               '-threads', str(threads), clip]
        r = subprocess.run(enc, input=frames.tobytes(), capture_output=True, timeout=timeout)
        if r.returncode:
            raise RuntimeError(f'ffmpeg encode failed: {r.stderr.decode(errors="replace")[-400:]}')
        dec = [exe, '-nostdin', '-hide_banner', '-loglevel', 'error', '-threads', str(threads), '-i', clip,
               '-f', 'rawvideo', '-pix_fmt', 'rgb24', 'pipe:1']
        r = subprocess.run(dec, capture_output=True, stdin=subprocess.DEVNULL, timeout=timeout)
        if r.returncode:
            raise RuntimeError(f'ffmpeg decode failed: {r.stderr.decode(errors="replace")[-400:]}')
    out = np.frombuffer(r.stdout, dtype=np.uint8)
    if out.size != n * h * w * 3:
        raise RuntimeError(f'H.264 round trip returned {out.size / (h * w * 3):g} frames for {n}')
    return out.reshape(n, h, w, 3).copy()


def _h264_video(job):
    """Pool worker (top level so it pickles): one video's frames in their given order."""
    paths, crf, exe, size, preset, fps = job
    frames = []
    for p in paths:
        with Image.open(p) as im:
            frames.append(np.asarray(im.convert('RGB').resize((size, size), Image.Resampling.BILINEAR),
                                     dtype=np.uint8))
    return h264_roundtrip(np.stack(frames), crf, exe, preset, fps)


def find_ffmpeg():
    """(exe, note). System ffmpeg first, then imageio-ffmpeg's binary; each must pass a
    tiny libx264 round trip. (None, reason) means the H.264 conditions are skipped."""
    candidates, notes = [], []
    if shutil.which('ffmpeg'):
        candidates.append(shutil.which('ffmpeg'))
    try:
        import imageio_ffmpeg
        candidates.append(imageio_ffmpeg.get_ffmpeg_exe())
    except Exception as e:
        notes.append(f'imageio_ffmpeg unavailable ({type(e).__name__})')
    probe = np.random.default_rng(0).integers(0, 256, (3, 32, 32, 3), dtype=np.uint8)
    for exe in candidates:
        try:
            h264_roundtrip(probe, 23, exe)
            version = subprocess.run([exe, '-version'], capture_output=True, stdin=subprocess.DEVNULL,
                                     timeout=30).stdout.decode(errors='replace').splitlines()[0]
            return exe, f'{exe} ({version})'
        except Exception as e:
            notes.append(f'{exe}: {e}')
    return None, 'no working ffmpeg with libx264: ' + ('; '.join(notes) or 'none found')


def h264_subset(test, n_per_class, seed):
    """Up to n real and n fake held-out videos, fixed by seed (all if fewer)."""
    v = test[['dataset', 'video_id', 'group_id', 'label']].drop_duplicates(['dataset', 'video_id'])
    parts = [g.sample(min(n_per_class, len(g)), random_state=seed) for _, g in v.groupby('label')]
    return pd.concat(parts).sort_values(['label', 'dataset', 'video_id']).reset_index(drop=True)


# ------------------------------------------------------------- statistics

def rank_auc(y, s):
    y = np.asarray(y).astype(bool)
    r = rankdata(s)
    n1 = int(y.sum())
    n0 = len(y) - n1
    if not n1 or not n0:
        return float('nan')
    return float((r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def weighted_auc(y, s, w=None):
    """Mann-Whitney AUC (ties count 1/2) for each row of weights w; one sort serves every
    bootstrap resample, since a resample is the original videos with integer weights."""
    y = np.asarray(y).astype(bool)
    s = np.asarray(s, dtype=np.float64)
    w = np.ones((1, len(s))) if w is None else np.atleast_2d(np.asarray(w, dtype=np.float64))
    order = np.argsort(s, kind='mergesort')
    s, y, w = s[order], y[order], w[:, order]
    starts = np.flatnonzero(np.r_[True, s[1:] != s[:-1]])
    pos = np.add.reduceat(w * y, starts, axis=1)
    neg = np.add.reduceat(w * ~y, starts, axis=1)
    below = np.cumsum(neg, axis=1) - neg
    num = (pos * (below + 0.5 * neg)).sum(1)
    den = pos.sum(1) * neg.sum(1)
    out = np.full(len(w), np.nan)
    ok = den > 0
    out[ok] = num[ok] / den[ok]
    return out


def cluster_weights(clusters, n_boot, seed):
    """(n_boot, n) integer weights: clusters drawn with replacement, every video of a
    drawn cluster counted once per draw."""
    codes, uniq = pd.factorize(pd.Series(list(clusters)))
    k = len(uniq)
    if k < 2:
        raise ValueError('Need at least two independent clusters')
    rng = np.random.default_rng(seed)
    counts = np.stack([np.bincount(rng.integers(0, k, k), minlength=k) for _ in range(n_boot)])
    return counts[:, codes]


def paired_bootstrap(a, b, n_boot=1000, seed=42):
    """AUC(a) - AUC(b) on identical videos; clusters = (dataset, group_id)."""
    keys = ['dataset', 'video_id', 'group_id', 'label']
    d = a.merge(b, on=keys, validate='one_to_one', suffixes=('_a', '_b'))
    if not len(d) == len(a) == len(b):
        raise ValueError('The two arms did not score exactly the same videos')
    y = d.label.to_numpy() == 1
    clusters = d.dataset.astype(str) + '||' + d.group_id.astype(str)
    w = cluster_weights(clusters, n_boot, seed)
    auc_a, auc_b = weighted_auc(y, d.probability_a)[0], weighted_auc(y, d.probability_b)[0]
    boot = weighted_auc(y, d.probability_a, w) - weighted_auc(y, d.probability_b, w)
    boot = boot[np.isfinite(boot)]
    if len(boot) < 0.9 * n_boot:
        raise ValueError('Too few valid bootstrap samples; inspect class/group balance')
    lo, hi = np.quantile(boot, [0.025, 0.975])
    return dict(auc_a=float(auc_a), auc_b=float(auc_b), delta=float(auc_a - auc_b), ci_low=float(lo),
                ci_high=float(hi), valid_bootstraps=int(len(boot)), videos=len(d),
                clusters=int(clusters.nunique()))


# ------------------------------------------------------- earlier outputs

def find_reuse_roots(skip=(), top='/kaggle/input', max_depth=11):
    """Folders holding <held>/<arm>/done.json from an earlier run, without listing face crops.
    A folder with images is pruned unless it has corpus subfolders: an earlier output keeps
    its plots next to <held>/."""
    skip, hits, top = {str(p) for p in skip}, set(), str(top)
    for dp, dns, fns in os.walk(top):
        p = Path(dp)
        if 'done.json' in fns and p.parent.name in CORPORA:
            hits.add(p.parent.parent)
        images = any(f.lower().endswith(IMG_EXT) for f in fns) and not set(dns) & set(CORPORA)
        if dp in skip or dp[len(top):].count(os.sep) >= max_depth or images:
            dns[:] = []
    return sorted(hits)


# ------------------------------------------------------------- experiment

class Experiment:
    """checkpoints: {'dual': {held: path}, 'rgb': {held: path}}; splits: {held: (train, val, test)}."""

    def __init__(self, cfg, splits, checkpoints, make_dual, make_rgb, preprocess, log, clock, out,
                 reuse_roots=()):
        self.cfg, self.splits, self.ckpt = cfg, splits, checkpoints
        self.builders = dict(dual=make_dual, rgb=make_rgb)
        self.preprocess, self.log, self.clock = preprocess, log, clock
        self.out = Path(out)
        self.out.mkdir(parents=True, exist_ok=True)
        self.reuse_roots = [Path(r) for r in reuse_roots]
        self.dev = cfg['device']
        self.cache = Path(cfg['h264_cache'])
        self.ffmpeg, self.ffmpeg_note = find_ffmpeg()
        self.conds = conditions(cfg, h264=self.ffmpeg is not None)
        self.log(f'H.264 conditions: {"ON, " if self.ffmpeg else "SKIPPED, "}{self.ffmpeg_note}')
        self.subsets, self.subset_frames = {}, {}
        for held, (_, _, test) in splits.items():
            vids = h264_subset(test, cfg['h264_videos_per_class'], cfg['h264_seed'])
            keep = set(zip(vids.dataset, vids.video_id))
            self.subsets[held] = vids
            self.subset_frames[held] = test[[k in keep for k in zip(test.dataset, test.video_id)]].reset_index(drop=True)
            (self.out / held).mkdir(exist_ok=True)
            vids.to_csv(self.out / held / 'h264_subset.csv', index=False)
        self.sha, self.stores, self.ratios, self.checks = {}, {}, {}, {}
        self.bench, self.h264_seconds, self.h264_failed = None, 0.0, {}
        if (self.out / 'preflight.json').is_file():
            self.bench = json.loads((self.out / 'preflight.json').read_text()).get('bench')

    # ---- models
    def load(self, kind, path):
        model = self.builders[kind]()
        state = torch.load(path, map_location='cpu', weights_only=True)
        if not isinstance(state, dict):
            raise TypeError(f'{path}: expected a plain state_dict')
        if state and all(k.startswith('net.') for k in state):   # Phase 11 SingleNet wrapper
            state = {k[4:]: v for k, v in state.items()}
        if kind == 'dual' and 'rgb.conv_stem.weight' not in state:
            raise ValueError(f'{path}: not a dual-stream state_dict')
        model.load_state_dict(state, strict=True)
        return model.to(self.dev)

    def mask_for(self, kind):
        return BlockDCTMask(drop_table(kind, self.cfg['max_drop']), self.cfg['seed'] + self.cfg['mask_seed_offset'])

    def checkpoint_sha(self, path):
        key = str(path)
        if key not in self.sha:
            self.sha[key] = file_sha256(path)
        return self.sha[key]

    def signature(self, held, name):
        spec = arm_spec(name)
        cfg = self.cfg
        return jsonable(dict(arm=name, definition=spec, held=held,
                             checkpoint_sha256=self.checkpoint_sha(self.ckpt[spec['model']][held]),
                             h264={k: cfg[k] for k in H264_KEYS}, **{k: cfg[k] for k in SIGNATURE_KEYS}))

    # ---- time
    def factor(self, kind, part):
        r = self.ratios.get((kind, part))
        return float(np.median(r)) if r else 1.0

    def observe(self, kind, part, actual, predicted):
        if predicted > 0 and actual > 0:
            self.ratios.setdefault((kind, part), []).append(actual / predicted)

    def projection(self, held, name, built=None, raw=False):
        """Seconds for training, evaluation and any H.264 encoding still to do; raw=True
        leaves out the correction learned from finished arms."""
        cfg, spec = self.cfg, arm_spec(name)
        kind = spec['model']
        b = (self.bench or DEFAULT_BENCH)[kind]
        tr, va, te = self.splits[held]
        sub = len(self.subset_frames[held]) if self.ffmpeg else 0
        n_eval = len(te) * (1 + len(cfg['eval_jpeg'])) + sub * len(cfg['h264_crf'])
        train = cfg['epochs'] * (len(tr) / b['train_img_s'] + len(va) / b['eval_img_s']) if spec['trained'] else 0.0
        built = set(self.stores) | set(self.h264_failed) if built is None else built
        todo = [c for c in cfg['h264_crf'] if (held, c) not in built] if self.ffmpeg else []
        h264 = len(todo) * len(self.subsets[held]) * H264_SEC_PER_VIDEO / cfg['h264_workers']
        ft, fe = (1.0, 1.0) if raw else (self.factor(kind, 'train'), self.factor(kind, 'eval'))
        return dict(train=train * ft, eval=n_eval / b['eval_img_s'] * fe, h264=h264)

    @staticmethod
    def hours_of(p):
        return 1.15 * sum(p.values()) / 3600

    # ---- reuse
    def find_done(self, held, name, sig):
        for root in [self.out] + self.reuse_roots:
            meta = root / held / name / 'done.json'
            if meta.is_file():
                try:
                    r = json.loads(meta.read_text())
                except ValueError:
                    continue
                if r.get('status') == 'complete' and r.get('signature') == sig:
                    return root, r
        return None, None

    def reuse(self, held, name, sig, d):
        root, r = self.find_done(held, name, sig)
        if r is None:
            return None
        if root != self.out:
            src = root / held / name
            for f in src.iterdir():
                if f.is_file():
                    shutil.copy2(f, d / f.name)
            r['reused_from'] = str(src)
            write_json(d / 'done.json', r)
            self.log(f'Reusing {name} | {held} held out from {src}')
        else:
            self.log(f'{name} | {held} held out: already complete in {d}')
        return r

    # ---- H.264
    def h264_store(self, held, crf):
        key = (held, crf)
        if key not in self.stores:
            frames = self.subset_frames[held]
            paths = frames.path.tolist()
            self.cache.mkdir(parents=True, exist_ok=True)
            stem = f'{self.out.name}_{held}_crf{crf}'
            arr_path, idx_path = self.cache / f'{stem}.npy', self.cache / f'{stem}.csv'
            if not (arr_path.is_file() and idx_path.is_file() and pd.read_csv(idx_path).path.tolist() == paths):
                self.build_h264(held, crf, frames, arr_path, idx_path)
            self.stores[key] = H264Store(np.load(arr_path, mmap_mode='r'), paths)
        return self.stores[key]

    def build_h264(self, held, crf, frames, arr_path, idx_path):
        cfg, t = self.cfg, time.time()
        s = cfg['h264_size']
        groups = list(frames.groupby(['dataset', 'video_id'], sort=False).indices.values())
        jobs = [(frames.path.iloc[ix].tolist(), crf, self.ffmpeg, s, cfg['h264_preset'], cfg['h264_fps'])
                for ix in groups]
        self.log(f'  H.264 CRF {crf} | {held}: encoding {len(jobs)} videos ({len(frames)} frames)')
        part = arr_path.with_name(arr_path.stem + '.partial.npy')
        arr = np.lib.format.open_memmap(part, mode='w+', dtype=np.uint8, shape=(len(frames), s, s, 3))
        try:
            # fork: the workers only read crops and run ffmpeg; they never touch CUDA.
            with multiprocessing.get_context('fork').Pool(cfg['h264_workers']) as pool:
                for k, (ix, decoded) in enumerate(zip(groups, pool.imap(_h264_video, jobs, chunksize=2))):
                    arr[ix] = decoded
                    if k % 200 == 0:
                        self.log(f'    H.264 CRF {crf} | {held}: video {k + 1}/{len(jobs)}')
            arr.flush()
        except BaseException:
            del arr
            part.unlink(missing_ok=True)
            self.h264_seconds += time.time() - t
            raise
        del arr
        os.replace(part, arr_path)
        pd.DataFrame(dict(path=frames.path)).to_csv(idx_path, index=False)
        sec = time.time() - t
        self.h264_seconds += sec
        write_json(self.out / held / f'h264_crf{crf}_build.json',
                   dict(held=held, crf=crf, videos=len(jobs), frames=len(frames), seconds=sec,
                        ffmpeg=self.ffmpeg_note, preset=cfg['h264_preset'], fps=cfg['h264_fps'], size=s,
                        pix_fmt='yuv420p', threads_per_clip=1))
        self.log(f'  H.264 CRF {crf} | {held}: done in {sec / 60:.1f} min')

    def clear_h264_cache(self):
        self.stores.clear()
        for f in self.cache.glob(f'{self.out.name}_*'):
            f.unlink()

    # ---- evaluation
    def missing(self, d):
        return [c for c in self.conds if not (d / f'pred_{c}.csv').is_file()]

    def scored(self, d):
        return [c for c in self.conds if (d / f'pred_{c}.csv').is_file()]

    def evaluate(self, model, held, name, d):
        cfg, test = self.cfg, self.splits[held][2]
        for cond in self.missing(d):
            t = time.time()
            what = f'{name} | {held} | {cond}'
            if cond == 'clean':
                pred = predict(model, test, self.preprocess, cfg, log=self.log, what=what)
            elif cond.startswith('jpeg'):
                pred = predict(model, test, self.preprocess, cfg, quality=int(cond[4:]), log=self.log, what=what)
            elif cond == 'clean_h264subset':
                clean = read_pred(d / 'pred_clean.csv')
                keep = set(zip(self.subsets[held].dataset, self.subsets[held].video_id))
                pred = clean[[k in keep for k in zip(clean.dataset, clean.video_id)]].reset_index(drop=True)
            elif cond.startswith('h264_crf'):
                crf = int(cond[8:])
                if (held, crf) in self.h264_failed:
                    continue
                try:
                    store = self.h264_store(held, crf)
                except Exception as e:
                    # Training and the other conditions stand; a later session adds this one.
                    if cfg['stop_on_error']:
                        raise
                    self.h264_failed[(held, crf)] = repr(e)
                    self.log(f'  H.264 CRF {crf} | {held}: FAILED, {cond} left out for this fold: {e!r}\n'
                             f'{traceback.format_exc()}')
                    continue
                pred = predict(model, self.subset_frames[held], self.preprocess, cfg, store=store,
                               log=self.log, what=what)
            else:
                raise ValueError(cond)
            pred.to_csv(d / f'pred_{cond}.csv', index=False)
            self.log(f'  {what}: AUC {macro_auc(pred):.4f} on {len(pred)} videos ({time.time() - t:.0f} s)')

    def consistency(self, held, name, d):
        auc = macro_auc(read_pred(d / 'pred_clean.csv'))
        ref = REFERENCE_AUC[name][held]
        if self.cfg['smoke']:
            verdict = 'not comparable (smoke frames)'
        else:
            verdict = 'PASS' if abs(auc - ref) <= 0.005 else f'WARN gap {auc - ref:+.4f}'
        self.checks[(held, name)] = dict(fold=held, arm=name, clean_auc=auc, reference_auc=ref,
                                         gap=auc - ref, verdict=verdict)
        pd.DataFrame(self.checks.values()).to_csv(self.out / 'consistency.csv', index=False)
        self.log(f'  consistency {name} | {held}: clean AUC {auc:.4f} vs Phase 13b {ref:.4f}: {verdict}')

    # ---- training
    def fit(self, held, name, d, proj):
        cfg, spec, dev = self.cfg, ARM_DEFS[name], self.dev
        train, val, _ = self.splits[held]
        seed_all(cfg['seed'], cfg['deterministic'])
        base = self.load(spec['model'], self.ckpt[spec['model']][held])
        for p in base.parameters():
            p.requires_grad_(True)
        model = MaskedFrequencyModel(base, self.mask_for(spec['mask'])).to(dev) if spec['mask'] else base
        opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
        scaler = torch.amp.GradScaler('cuda', enabled=cfg['amp'])
        n_fake = int((train.label == 1).sum())
        pos_weight = torch.tensor([(len(train) - n_fake) / max(1, n_fake)], device=dev)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        ds = FrameData(train, self.preprocess, seed=cfg['seed'] + cfg['realjpeg_seed_offset'],
                       realjpeg=(cfg['realjpeg_p'], tuple(cfg['realjpeg_q'])) if spec['realjpeg'] else None,
                       subsampling=cfg['jpeg_subsampling'])
        dl = loader(ds, cfg, train=True)
        rows, best, best_epoch, status = [], -float('inf'), None, 'complete'
        settled, warmup, t_all = False, 0, 0.0
        eval_h = 1.15 * (proj['eval'] + proj['h264']) / 3600
        for epoch in range(1, cfg['epochs'] + 1):
            ds.set_epoch(epoch)
            model.train()
            t, losses, skipped = time.time(), [], 0
            for step, (x, y, _) in enumerate(dl):
                loss, overflow = train_step(model, x.to(dev, non_blocking=True),
                                            y.float().to(dev, non_blocking=True), criterion, opt, scaler, cfg)
                losses.append(loss)
                if overflow and not settled:
                    warmup += 1   # GradScaler's initial descent from 2**16, skipped by design
                elif overflow:
                    skipped += 1
                else:
                    settled = True
                if step % cfg['log_every'] == 0:
                    self.log(f'  {name} | {held} | epoch {epoch} step {step + 1}/{len(dl)} '
                             f'loss {np.mean(losses[-50:]):.4f} scale {scaler.get_scale():g}')
            if warmup > WARMUP_SKIP_LIMIT:
                raise FloatingPointError(f'{name}/{held}: {warmup} overflowing steps before the first finite one')
            check_overflow(skipped, len(dl), f'{name}/{held} epoch {epoch}', cfg['max_overflow_frac'])
            train_sec = time.time() - t
            t = time.time()
            vp = predict(model, val, self.preprocess, cfg, log=self.log, what=f'{name} | {held} | source val')
            auc, val_sec = macro_auc(vp), time.time() - t
            t_all += train_sec + val_sec
            rows.append(dict(fold=held, arm=name, epoch=epoch, train_loss=float(np.mean(losses)),
                             source_val_auc=auc, steps=len(dl), overflow_steps=skipped,
                             warmup_skipped_steps=warmup, grad_scale=scaler.get_scale(),
                             train_seconds=train_sec, val_seconds=val_sec))
            pd.DataFrame(rows).to_csv(d / 'training.csv', index=False)
            self.log(f'  {name} | {held} | epoch {epoch}: loss {np.mean(losses):.4f} | source-val AUC {auc:.4f} '
                     f'| overflow {skipped} (+{warmup} warm-up) | {train_sec / 60:.1f}+{val_sec / 60:.1f} min')
            if auc > best:
                best, best_epoch = auc, epoch
                torch.save(base.state_dict(), d / 'best.pth')   # base only: the hook is not saved
                vp.to_csv(d / 'pred_val_best.csv', index=False)
            per_epoch_h = (train_sec + val_sec) / 3600
            if epoch < cfg['epochs'] and self.clock() + per_epoch_h + eval_h > cfg['time_limit_h']:
                status = 'truncated'
                self.log(f'  {name} | {held}: time limit after epoch {epoch}; scoring the best epoch so far')
                break
        if spec['mask'] and model.mask.calls == 0:
            raise RuntimeError('Mask hook never executed')
        del model, base, opt, scaler
        free()
        return dict(status=status, best_epoch=best_epoch, source_val_auc=best, epochs_run=len(rows),
                    train_seconds=t_all)

    # ---- one arm on one fold
    def run_unit(self, held, name):
        cfg, spec = self.cfg, arm_spec(name)
        kind = spec['model']
        d = self.out / held / name
        d.mkdir(parents=True, exist_ok=True)
        sig = self.signature(held, name)
        t0 = time.time()
        done = self.reuse(held, name, sig, d)
        if done is not None:
            todo = self.missing(d)
            src = d / 'best.pth' if spec['trained'] else self.ckpt[kind][held]
            if todo and Path(src).is_file():
                self.log(f'  {name} | {held}: adding conditions {todo}')
                model = self.load(kind, src)
                self.evaluate(model, held, name, d)
                del model
                done['conditions'] = self.scored(d)
                write_json(d / 'done.json', done)
            elif todo:
                self.log(f'  {name} | {held}: {todo} missing and no weights to score them')
        else:
            p, p_raw = self.projection(held, name), self.projection(held, name, raw=True)
            need = self.hours_of(p)
            if self.clock() + need > cfg['time_limit_h']:
                self.log(f'SKIP {name} | {held}: needs ~{need:.2f} h, {cfg["time_limit_h"] - self.clock():.2f} h '
                         f'left. Run it in a new session with this output attached (reuse).')
                done = dict(status='skipped_time_budget', fold=held, arm=name, projected_hours=need,
                            at_hours=self.clock(), signature=sig)
                write_json(d / 'done.json', done)
                return done
            self.log(f'=== {name} | {held} held out | ~{need:.2f} h projected ===')
            for f in d.iterdir():   # leftovers of an unfinished attempt must not count as evaluated
                if f.is_file() and (f.name.startswith('pred_') or f.name in ('training.csv', 'best.pth')):
                    f.unlink()
            info = dict(status='complete', best_epoch=None, source_val_auc=None, train_seconds=0.0)
            if spec['trained']:
                info = self.fit(held, name, d, p)
                if info['status'] == 'complete':
                    self.observe(kind, 'train', info['train_seconds'], p_raw['train'])
                model = self.load(kind, d / 'best.pth')
            else:
                model = self.load(kind, self.ckpt[kind][held])
            t, h0 = time.time(), self.h264_seconds
            self.evaluate(model, held, name, d)
            eval_sec = time.time() - t - (self.h264_seconds - h0)
            self.observe(kind, 'eval', eval_sec, p_raw['eval'])
            del model
            done = dict(info, fold=held, arm=name, seconds=time.time() - t0, eval_seconds=eval_sec,
                        projected_hours=need, finished_at_hours=self.clock(), conditions=self.scored(d),
                        missing_conditions=self.missing(d), signature=sig)
            write_json(d / 'done.json', done)
            self.log(f'  {name} | {held}: {done["status"]} in {done["seconds"] / 60:.1f} min')
        if not spec['trained'] and (d / 'pred_clean.csv').is_file():
            self.consistency(held, name, d)
        free()
        return done

    def run(self, arms, folds, originals=True):
        """Arm-major order: if time runs out, the late (lower-priority) arms are the ones
        missing, never a whole fold of the primary arms."""
        names = (list(ORIGINAL_DEFS) if originals else []) + list(arms)
        for n in names:
            arm_spec(n)
        if self.bench is None:
            self.log('No preflight benchmark: projections use DEFAULT_BENCH until arms finish')
        records = []
        for name in names:
            for held in folds:
                try:
                    r = self.run_unit(held, name)
                except Exception as e:
                    if self.cfg['stop_on_error']:
                        raise
                    self.log(f'FAILED {name} | {held}: {e!r}\n{traceback.format_exc()}')
                    r = dict(status='failed', fold=held, arm=name, error=repr(e))
                    write_json(self.out / held / name / 'done.json', r)
                    try:
                        free()
                    except Exception:
                        pass
                records.append({k: r.get(k) for k in ('fold', 'arm', 'status', 'best_epoch', 'source_val_auc',
                                                     'seconds', 'projected_hours', 'reused_from', 'error')})
                pd.DataFrame(records).to_csv(self.out / 'status.csv', index=False)
        try:
            self.clear_h264_cache()
        except Exception as e:
            self.log(f'clear_h264_cache failed: {e!r}')
        try:
            self._write_metadata(arms, folds, originals, records)
        except Exception as e:
            self.log(f'run_metadata.json not written: {e!r}')
        return pd.DataFrame(records)

    def _write_metadata(self, arms, folds, originals, records):
        write_json(self.out / 'run_metadata.json', dict(
            smoke=self.cfg['smoke'], arms=list(arms), folds=list(folds), originals=originals,
            config=self.cfg, torch=torch.__version__, numpy=np.__version__,
            gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
            ffmpeg=self.ffmpeg_note, reuse_roots=self.reuse_roots, bench=self.bench,
            checkpoints={k: {h: dict(path=str(p), sha256=self.checkpoint_sha(p)) for h, p in v.items() if h in folds}
                         for k, v in self.ckpt.items()},
            time_ratios={f'{k}/{p}': v for (k, p), v in self.ratios.items()},
            consistency=list(self.checks.values()), wall_hours=self.clock(), status=records,
            h264_failed={f'{h}/crf{c}': e for (h, c), e in self.h264_failed.items()}))

    # ---- preflight on the real checkpoints
    def preflight(self, arms, folds, originals=True, held=None):
        cfg, dev = self.cfg, self.dev
        held = held or folds[0]
        train, val, _ = self.splits[held]
        seed_all(cfg['seed'], cfg['deterministic'])
        dual = self.load('dual', self.ckpt['dual'][held]).eval()
        rgb = self.load('rgb', self.ckpt['rgb'][held]).eval()
        self.log(f'PASS: strict load of {Path(self.ckpt["dual"][held]).name} and {Path(self.ckpt["rgb"][held]).name}')
        head = train.groupby('label', group_keys=False).head(2)
        x = torch.stack([self.preprocess(_open_rgb(p)) for p in head.path]).to(dev)
        y = torch.tensor(head.label.to_numpy(), dtype=torch.float32, device=dev)
        with torch.no_grad():
            ref = dual(x).float()
            for kind in MASK_KINDS:
                w = MaskedFrequencyModel(dual, self.mask_for(kind)).to(dev).eval()
                gap = (w(x).float() - ref).abs().max().item()
                w.remove()
                assert gap <= 1e-6, f'{kind}: eval logits changed by {gap} with the mask disabled'
            assert torch.isfinite(rgb(x)).all()
        self.log('PASS: masks in eval mode leave the dual-stream logits identical')
        del dual, rgb
        free()

        m = self.load('dual', self.ckpt['dual'][held]).train()   # BN uses batch stats: same input, same output
        seen = {}
        hook = m.freq.register_forward_hook(lambda mod, i, o: seen.__setitem__('f', o.detach().clone()))
        with torch.no_grad():
            m(x)
            rgb_ref, freq_ref = getattr(m, '_pooled', None), seen['f']
            rgb_ref = None if rgb_ref is None else rgb_ref.clone()
            w = MaskedFrequencyModel(m, self.mask_for('qtable')).to(dev).train()
            w(x)
        hook.remove()
        assert w.mask.calls == 1, 'FAD hook did not run'
        assert not torch.allclose(seen['f'], freq_ref), 'mask did not change the frequency stream'
        if rgb_ref is not None:
            assert torch.allclose(m._pooled, rgb_ref, atol=1e-5), 'mask changed the RGB stream'
        self.log('PASS: training mask changes the frequency stream and leaves the RGB stream identical')
        del m, w
        free()

        for kind, mask in [('dual', None)] + [('dual', k) for k in MASK_KINDS] + [('rgb', None)]:
            base = self.load(kind, self.ckpt[kind][held])
            for p in base.parameters():
                p.requires_grad_(True)
            model = MaskedFrequencyModel(base, self.mask_for(mask)).to(dev) if mask else base
            model.train()
            with torch.autocast('cuda', enabled=cfg['amp']):
                loss = nn.functional.binary_cross_entropy_with_logits(model(x).reshape(-1).float(), y)
            loss.backward()
            grads = [p.grad for p in model.parameters() if p.grad is not None]
            assert torch.isfinite(loss) and grads and all(torch.isfinite(g).all() for g in grads), \
                f'{kind}/{mask}: non-finite loss or gradients'
            if mask:
                assert model.mask.calls > 0, 'FAD hook did not run'
            opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
            _, overflow = train_step(model, x, y, nn.BCEWithLogitsLoss(), opt,
                                     torch.amp.GradScaler('cuda', enabled=cfg['amp']), cfg)
            self.log(f'PASS: {kind} mask={mask}: AMP loss {loss.item():.4f}, finite gradients; '
                     f'GradScaler step {"skipped (fp16 overflow at scale 2**16, by design)" if overflow else "taken"}')
            del model, base, opt, grads
            free()

        bench = {kind: self.benchmark(kind, mask, held) for kind, mask in (('dual', 'qtable'), ('rgb', None))}
        self.bench = bench
        for kind, b in bench.items():
            self.log(f'  speed {kind}: train {b["train_img_s"]:.0f} img/s (real-JPEG data on), '
                     f'eval {b["eval_img_s"]:.0f} img/s (JPEG q50, after the first batch)')

        names = (list(ORIGINAL_DEFS) if originals else []) + list(arms)
        rows, now, built = [], self.clock(), set()
        for name in names:
            for h in folds:
                root, _ = self.find_done(h, name, self.signature(h, name))
                if root is not None:
                    rows.append(dict(arm=name, fold=h, train_h=0.0, eval_h=0.0, hours=0.0, action=f'reuse {root}'))
                    continue
                p = self.projection(h, name, built)
                hrs = self.hours_of(p)
                fits = now + hrs <= cfg['time_limit_h']
                if fits:
                    now += hrs
                    built |= {(h, c) for c in cfg['h264_crf']}
                rows.append(dict(arm=name, fold=h, train_h=1.15 * p['train'] / 3600,
                                 eval_h=1.15 * (p['eval'] + p['h264']) / 3600, hours=hrs,
                                 action='run' if fits else 'SKIP (time budget)'))
        plan = pd.DataFrame(rows)
        print(plan.round(2).to_string(index=False))
        total = self.clock() + plan.hours.sum()
        print(f'Projected finish: {total:.2f} h of this session (limit {cfg["time_limit_h"]} h; '
              f'{self.clock():.2f} h used so far).')
        advice = ''
        if total > cfg['time_limit_h']:
            drop, left = [], total
            for name in reversed([n for n in arms]):
                if left <= cfg['time_limit_h']:
                    break
                drop.append(name)
                left -= plan[plan.arm == name].hours.sum()
            advice = (f'Over the limit. Drop these arms from ARMS (lowest priority first): {drop} '
                      f'(projected {left:.2f} h), or run them in a second session with this output attached '
                      f'(finished arms are reused). Without changes, arms marked SKIP are skipped.')
            print(advice)
        else:
            print('Fits within the limit with every selected fold and arm.')
        result = dict(bench=bench, held=held, plan=plan.to_dict('records'), projected_finish_h=total,
                      advice=advice, ffmpeg=self.ffmpeg_note)
        write_json(self.out / 'preflight.json', result)
        return result

    def benchmark(self, kind, mask, held):
        """Train speed includes data loading with real-JPEG on (worst case)."""
        cfg, dev = self.cfg, self.dev
        train, val, _ = self.splits[held]
        seed_all(cfg['seed'], cfg['deterministic'])
        base = self.load(kind, self.ckpt[kind][held])
        model = MaskedFrequencyModel(base, self.mask_for(mask)).to(dev) if mask else base
        opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
        scaler = torch.amp.GradScaler('cuda', enabled=cfg['amp'])
        ds = FrameData(train, self.preprocess, realjpeg=(cfg['realjpeg_p'], tuple(cfg['realjpeg_q'])),
                       seed=cfg['seed'], subsampling=cfg['jpeg_subsampling'])
        ds.set_epoch(1)
        dl = loader(ds, cfg, train=True)
        n = min(len(dl), cfg['bench_batches'])
        if n < 3:
            raise ValueError('Need at least three training batches to benchmark')
        warm, t, imgs = 2, None, 0
        model.train()
        for i, (x, y, _) in enumerate(dl):
            train_step(model, x.to(dev, non_blocking=True), y.float().to(dev, non_blocking=True),
                       nn.BCEWithLogitsLoss(), opt, scaler, cfg)
            if i == warm - 1:
                torch.cuda.synchronize()
                t = time.time()
            elif i >= warm:
                imgs += len(x)
            if i + 1 >= n:
                break
        torch.cuda.synchronize()
        train_ips = imgs / (time.time() - t)
        del dl
        # Eval with JPEG q50 in the workers (conservative for clean and H.264); timed after the
        # first batch so worker start-up does not halve the figure, as it did in Phase 13.
        frames = val.head(min(len(val), cfg['eval_batch_size'] * 40))
        dl = loader(FrameData(frames, self.preprocess, quality=50, subsampling=cfg['jpeg_subsampling']), cfg)
        model.eval()
        t0, t, imgs = time.time(), None, 0
        with torch.no_grad():
            for k, (x, _, _) in enumerate(dl):
                with torch.autocast(dev.split(':')[0], dtype=torch.float16, enabled=cfg['amp']):
                    model(x.to(dev, non_blocking=True)).float().cpu()
                if k == 0:
                    t = time.time()
                else:
                    imgs += len(x)
        del dl
        eval_ips = imgs / (time.time() - t) if imgs else len(frames) / (time.time() - t0)
        del model, base, opt
        free()
        return dict(train_img_s=float(train_ips), eval_img_s=float(eval_ips))


# ---------------------------------------------------------------- summary

def collect(out):
    out = Path(out)
    rows, preds, hist = [], {}, []
    for held in [h for h in CORPORA if (out / h).is_dir()]:
        for d in sorted(p for p in (out / held).iterdir() if p.is_dir()):
            meta = d / 'done.json'
            status = json.loads(meta.read_text()).get('status') if meta.is_file() else 'incomplete'
            if status not in ('complete', 'truncated'):
                continue
            for f in sorted(d.glob('pred_*.csv')):
                cond = f.stem[len('pred_'):]
                if cond == 'val_best':
                    continue
                p = read_pred(f)
                preds[(held, d.name, cond)] = p
                rows.append(dict(fold=held, arm=d.name, condition=cond, auc=macro_auc(p), videos=len(p),
                                 real=int((p.label == 0).sum()), fake=int((p.label == 1).sum()), status=status,
                                 file=str(f.relative_to(out))))
            if (d / 'training.csv').is_file():
                hist.append(pd.read_csv(d / 'training.csv'))
    return pd.DataFrame(rows), preds, (pd.concat(hist, ignore_index=True) if hist else pd.DataFrame())


def _order(frame, cond_order):
    f = frame.copy()
    f['_f'] = f.fold.map({h: i for i, h in enumerate(CORPORA)}) if 'fold' in f else 0
    f['_a'] = f.arm.map({a: i for i, a in enumerate(ARM_ORDER)}).fillna(99) if 'arm' in f else 0
    f['_c'] = f.condition.map({c: i for i, c in enumerate(cond_order)}).fillna(99)
    f['_k'] = (f.comparison.map({f'{a} - {b}': i for i, (a, b) in enumerate(COMPARISONS)}).fillna(99)
               if 'comparison' in f else 0)
    keys = ['_f', '_a', '_c', '_k'] if 'comparison' not in f else ['_k', '_f', '_c']
    return f.sort_values(keys).drop(columns=['_f', '_a', '_c', '_k']).reset_index(drop=True)


def summarise(out, cfg, log=print):
    """Every number comes from the pred_*.csv and training.csv files under `out`."""
    out = Path(out)
    cond_order = conditions(cfg, h264=True)
    results, preds, hist = collect(out)
    if results.empty:
        log('No finished arms under ' + str(out))
        return dict(results=results)
    results = _order(results, cond_order)
    results.to_csv(out / 'results.csv', index=False)
    wide = results.pivot_table(index=['fold', 'arm'], columns='condition', values='auc')
    wide = wide[[c for c in cond_order if c in wide.columns]]
    wide = wide.reindex(sorted(wide.index, key=lambda k: (CORPORA.index(k[0]), ARM_ORDER.index(k[1])
                                                          if k[1] in ARM_ORDER else 99)))
    wide.to_csv(out / 'results_wide.csv')

    mean = (results.groupby(['arm', 'condition'])
            .agg(mean_auc=('auc', 'mean'), folds=('fold', 'size'), fold_list=('fold', lambda s: ','.join(s)),
                 truncated_folds=('status', lambda s: int((s == 'truncated').sum())))
            .reset_index())
    mean = _order(mean, cond_order)
    mean.to_csv(out / 'mean_over_folds.csv', index=False)

    # 'truncated' = the time guard stopped training early; such rows are flagged, never silently mixed in.
    status = {(r.fold, r.arm): r.status for r in results.itertuples()}
    rows = []
    for held in [h for h in CORPORA if h in set(results.fold)]:
        log(f'Paired bootstrap | {held} held out')
        for cond in cond_order:
            for a, b in COMPARISONS:
                pa, pb = preds.get((held, a, cond)), preds.get((held, b, cond))
                if pa is None or pb is None:
                    continue
                sa, sb = status[(held, a)], status[(held, b)]
                row = dict(fold=held, condition=cond, comparison=f'{a} - {b}', arm_a=a, arm_b=b,
                           endpoint=ENDPOINTS.get((a, b, cond), ''), status_a=sa, status_b=sb,
                           truncated_arm='truncated' in (sa, sb))
                if row['endpoint'] and row['truncated_arm']:
                    log(f'WARNING: endpoint {a} - {b} @ {cond} | {held}: one arm is truncated ({sa} / {sb})')
                try:
                    row.update(paired_bootstrap(pa, pb, cfg['bootstrap'], cfg['seed']))
                except ValueError as e:
                    row.update(error=str(e))
                rows.append(row)
    deltas = pd.DataFrame(rows)
    if len(deltas):
        deltas = _order(deltas, cond_order)
    if 'ci_low' in deltas:
        deltas['ci_excludes_0'] = (deltas.ci_low > 0) | (deltas.ci_high < 0)
    deltas.to_csv(out / 'paired_deltas.csv', index=False)
    endpoints = deltas[deltas.endpoint != ''] if len(deltas) else deltas
    endpoints.to_csv(out / 'endpoints.csv', index=False)
    mean_deltas = pd.DataFrame()
    if len(deltas) and 'delta' in deltas:
        mean_deltas = (deltas.dropna(subset=['delta']).groupby(['comparison', 'condition', 'endpoint'])
                       .agg(mean_delta=('delta', 'mean'), folds=('fold', 'size'),
                            folds_with_truncated_arm=('truncated_arm', 'sum'),
                            folds_ci_above_0=('ci_low', lambda s: int((s > 0).sum())),
                            folds_ci_below_0=('ci_high', lambda s: int((s < 0).sum())))
                       .reset_index())
        mean_deltas = _order(mean_deltas, cond_order)
    mean_deltas.to_csv(out / 'paired_deltas_mean.csv', index=False)
    write_json(out / 'summary_metadata.json', dict(
        bootstrap=cfg['bootstrap'], seed=cfg['seed'], clusters='(dataset, group_id)',
        delta='AUC(arm_a) - AUC(arm_b) on identical videos; 95% percentile interval',
        endpoints={f'{a} - {b} @ {c}': v for (a, b, c), v in ENDPOINTS.items()},
        h264_note='h264_* and clean_h264subset use the same fixed video subset (h264_subset.csv per fold)',
        mean_note='mean_over_folds and paired_deltas_mean are point estimates; check the folds column',
        truncated_note='status truncated = training stopped early by the time guard; see truncated_arm, '
                       'status_a/status_b and truncated_folds before reading a delta'))
    make_plots(results, hist, out, cond_order)
    return dict(results=results, wide=wide, mean=mean, deltas=deltas, endpoints=endpoints,
                mean_deltas=mean_deltas, history=hist)


def make_plots(results, hist, out, cond_order):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    for held, g in results.groupby('fold'):
        conds = [c for c in cond_order if c in set(g.condition)]
        fig, ax = plt.subplots(figsize=(9, 4.8))
        for arm in [a for a in ARM_ORDER if a in set(g.arm)]:
            v = g[g.arm == arm].set_index('condition').auc.reindex(conds)
            ax.plot(conds, v.values, marker='o', label=arm, ls='--' if arm in ORIGINAL_DEFS else '-')
        ax.axhline(0.5, color='grey', lw=0.8, ls=':')
        ax.set(ylabel='video-level AUC', title=f'{held} held out: robustness (H.264 columns on the fixed subset)')
        ax.tick_params(axis='x', rotation=20)
        ax.legend(fontsize=7, ncol=2)
        fig.tight_layout()
        fig.savefig(out / f'robustness_{held}.png', dpi=160)
        plt.close(fig)
    if hist.empty:
        return
    folds = [h for h in CORPORA if h in set(hist.fold)]
    fig, axes = plt.subplots(len(folds), 2, figsize=(10, 3.2 * len(folds)), squeeze=False)
    for r, held in enumerate(folds):
        for arm, g in hist[hist.fold == held].groupby('arm'):
            axes[r, 0].plot(g.epoch, g.train_loss, marker='o', label=arm)
            axes[r, 1].plot(g.epoch, g.source_val_auc, marker='o', label=arm)
        axes[r, 0].set(xlabel='epoch', ylabel='train BCE', title=f'{held} held out')
        axes[r, 1].set(xlabel='epoch', ylabel='source-val macro AUC', title=f'{held} held out')
    axes[0, 1].legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(out / 'training_curves.png', dpi=160)
    plt.close(fig)


def zip_out(out):
    """Everything except checkpoints (.pth stays in /kaggle/working for the version output)."""
    faulthandler.cancel_dump_traceback_later()   # the run is finished: disarm the hang watchdog
    out = Path(out)
    archive = out.with_suffix('.zip')
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in sorted(out.rglob('*')):
            if p.is_file() and p.suffix != '.pth':
                z.write(p, p.relative_to(out.parent))
    return archive


# ---------------------------------------------------------------- testing

def _to_tensor(image):
    return torch.from_numpy(np.asarray(image, dtype=np.float32).copy()).permute(2, 0, 1) / 255


def unit_tests(log=print):
    """CPU, synthetic data, no downloads."""
    torch.manual_seed(7)
    x = torch.randn(2, 3, 16, 24, requires_grad=True)
    m = BlockDCTMask(drop_table('qtable', 0.7), seed=1)
    c = m.coefficients(x)
    assert torch.allclose(m.reconstruct(c), x, atol=5e-6), 'DCT round trip'
    m.eval()
    assert m(x) is x, 'evaluation must disable masking'
    m.train()
    y = m(x)
    assert y.shape == x.shape and torch.isfinite(y).all() and not torch.allclose(x, y)
    assert torch.allclose(m.coefficients(y)[..., 0, 0], c[..., 0, 0], atol=1e-5), 'DC changed'
    assert m.coefficients(y).square().sum() <= c.square().sum() + 1e-3, 'energy increased'
    y.square().mean().backward()
    assert torch.isfinite(x.grad).all()
    log('PASS: DCT round trip, eval identity, DC preserved, energy, gradients')

    ac = np.ones((8, 8), bool)
    ac[0, 0] = False
    q = np.array(JPEG_LUMA, float)
    tq, tu, ti = (drop_table(k, 0.35) for k in MASK_KINDS)
    means = [t[ac].mean() for t in (tq, tu, ti)]
    assert max(means) - min(means) < 1e-6 and all(t[0, 0] == 0 for t in (tq, tu, ti))
    assert tq[7, 7] > tq[0, 1] and ti[0, 1] > ti[7, 7] and np.ptp(tu[ac]) == 0
    o = np.argsort(q[ac], kind='stable')
    assert (np.diff(tq[ac][o]) >= 0).all() and (np.diff(ti[ac][o]) <= 1e-12).all(), 'inverted order'
    assert np.allclose(np.sort(ti[ac]).sum(), np.sort(tq[ac]).sum())
    for t in (tq, tu, ti):
        keep = BlockDCTMask(t, 3).keep_mask(20000, torch.device('cpu'))[:, 0, 0, 0].float().mean(0).numpy()
        assert np.abs((1 - keep) - t).max() < 0.02, 'empirical drop rate differs from the table'
    try:
        drop_table('flat')
    except ValueError:
        pass
    else:
        raise AssertionError('unknown mask kind accepted')
    log(f'PASS: qmask/randmask/invmask mean AC drop {means[0]:.6f} each, orderings, empirical rates')

    a, b = BlockDCTMask(tq, 8).train(), BlockDCTMask(tq, 8).train()
    assert torch.equal(a(x.detach()), b(x.detach())), 'mask RNG not reproducible'
    state = torch.random.get_rng_state()
    a(x.detach())
    assert torch.equal(state, torch.random.get_rng_state()), 'mask moved the global RNG'

    class Toy(nn.Module):
        def __init__(self):
            super().__init__()
            self.rgb, self.fad = nn.Identity(), nn.Identity()

        def forward(self, z):
            return self.rgb(z), self.fad(z)

    w = MaskedFrequencyModel(Toy(), BlockDCTMask(tq, 1))
    rgb, freq = w(x.detach())
    assert torch.equal(rgb, x.detach()) and not torch.equal(freq, x.detach()), 'branch isolation'
    w.eval()
    rgb, freq = w(x.detach())
    assert torch.equal(rgb, freq)
    w.train()
    w.base.eval()   # validation scored through the bare base must not be masked either
    rgb, freq = w(x.detach())
    assert torch.equal(rgb, freq), 'hook active while base is in eval mode'
    w.train()
    w.remove()
    w.train()
    rgb, freq = w(x.detach())
    assert torch.equal(rgb, freq), 'hook not removed'
    log('PASS: independent mask RNG, branch isolation through the fad hook')

    with tempfile.TemporaryDirectory() as root:
        rng = np.random.default_rng(0)
        paths = []
        (Path(root) / 'crops').mkdir()
        for i in range(3):
            p = Path(root) / 'crops' / f'{i}.png'
            Image.fromarray(rng.integers(0, 256, (48, 40, 3), dtype=np.uint8)).save(p)
            paths.append(str(p))
        frame = pd.DataFrame(dict(path=paths, label=[0, 1, 0]))
        ds = FrameData(frame, _to_tensor, realjpeg=(1.0, (20, 95)), seed=5)
        ds.set_epoch(1)
        first = ds[0][0]
        ds2 = FrameData(frame, _to_tensor, realjpeg=(1.0, (20, 95)), seed=5)
        ds2.set_epoch(1)
        assert torch.equal(first, ds[0][0]) and torch.equal(first, ds2[0][0]), 'real-JPEG not deterministic'
        params = {realjpeg_params(5, e, 0, 0.5, (20, 95)) for e in range(10)}
        assert len(params) > 1, 'real-JPEG draws do not change across epochs'
        ds.set_epoch(2)
        assert any(not torch.equal(ds[i][0], ds2[i][0]) for i in range(3)), 'real-JPEG identical across epochs'
        hits = np.mean([realjpeg_params(5, 1, i, 0.5, (20, 95))[0] for i in range(4000)])
        qs = [realjpeg_params(5, 1, i, 1.0, (20, 95))[1] for i in range(4000)]
        assert abs(hits - 0.5) < 0.03 and min(qs) == 20 and max(qs) == 95
        clean = FrameData(frame, _to_tensor)[1][0]
        assert clean.shape == (3, 48, 40) and not torch.equal(clean, FrameData(frame, _to_tensor, quality=20)[1][0])

        tree = Path(root) / 'out' / 'phase14b' / 'dfdc' / 'control'
        tree.mkdir(parents=True)
        (tree / 'done.json').write_text('{}')
        shutil.copy(paths[0], tree.parent.parent / 'robustness_dfdc.png')   # Cell 8 plots sit next to <held>/
        hidden = Path(root) / 'crops' / 'sub' / 'dfdc' / 'control'          # below an image folder: never walked
        hidden.mkdir(parents=True)
        (hidden / 'done.json').write_text('{}')
        assert find_reuse_roots(top=root) == [Path(root) / 'out' / 'phase14b'], 'reuse discovery'
    log('PASS: real-JPEG augmentation deterministic per (seed, epoch, index), varies by epoch; reuse discovery')

    dev = 'cuda' if torch.cuda.is_available() else 'cpu'   # the run's own GradScaler is the CUDA one
    toy = nn.Linear(4, 1).to(dev)
    opt = torch.optim.SGD(toy.parameters(), lr=0.1)
    scaler = torch.amp.GradScaler(dev, init_scale=8.0)
    tcfg = dict(device=dev, amp=False, grad_clip=1.0)
    xb, yb = torch.randn(6, 4, device=dev), torch.tensor([0., 1., 0., 1., 1., 0.], device=dev)
    hook = toy.weight.register_hook(lambda g: g * float('inf'))
    before = toy.weight.detach().clone()
    _, overflow = train_step(toy, xb, yb, nn.BCEWithLogitsLoss(), opt, scaler, tcfg)
    assert overflow and torch.equal(toy.weight.detach(), before) and scaler.get_scale() < 8.0, \
        'an inf gradient must skip the step, not raise'
    hook.remove()
    _, overflow = train_step(toy, xb, yb, nn.BCEWithLogitsLoss(), opt, scaler, tcfg)
    assert not overflow and not torch.equal(toy.weight.detach(), before)
    check_overflow(3, 12, 'test')
    check_overflow(28, 562, 'test')
    for bad in ((4, 12), (29, 562)):
        try:
            check_overflow(*bad, 'test')
        except FloatingPointError:
            pass
        else:
            raise AssertionError(f'{bad[0]} of {bad[1]} overflowing steps accepted')
    log('PASS: GradScaler overflow path skips the step; more than max(5%, 3) overflows raises')

    rng = np.random.default_rng(1)
    for _ in range(5):
        yy = rng.integers(0, 2, 300)
        ss = np.round(rng.normal(size=300) + yy * 0.7, 1)   # rounded: many ties
        sk = roc_auc_score(yy, ss)
        assert abs(rank_auc(yy, ss) - sk) < 1e-12 and abs(weighted_auc(yy, ss)[0] - sk) < 1e-12
        cnt = rng.integers(0, 3, 300)
        rep = np.repeat(np.arange(300), cnt)
        assert abs(weighted_auc(yy, ss, cnt)[0] - roc_auc_score(yy[rep], ss[rep])) < 1e-12
    v = pd.DataFrame(dict(dataset='dfdc', video_id=[f'v{i}' for i in range(60)],
                          group_id=[f'g{i // 2}' for i in range(60)], label=rng.integers(0, 2, 60)))
    v['probability'] = rng.uniform(size=60)
    r = paired_bootstrap(v, v, 200, 42)
    assert r['delta'] == 0 and r['ci_low'] == 0 and r['ci_high'] == 0, 'identical arms must give exactly 0'
    v2 = v.assign(probability=np.clip(v.probability + 0.3 * v.label, 0, 1))
    r2 = paired_bootstrap(v2, v, 200, 42)
    assert r2['delta'] > 0 and r2['clusters'] == 30 and r2['ci_low'] <= r2['delta'] <= r2['ci_high']
    sub = h264_subset(pd.concat([v] * 2, ignore_index=True), 5, 42)
    assert sub.equals(h264_subset(v, 5, 42)) and (sub.label.value_counts() == 5).all()
    log('PASS: rank AUC = sklearn (with ties and integer weights), paired bootstrap identical arms = 0, '
        'H.264 subset fixed by seed')

    exe, note = find_ffmpeg()
    if exe is None:
        log(f'SKIP: H.264 round trip test ({note}); the H.264 conditions will be skipped')
    else:
        g = np.linspace(0, 255, 224)
        base = np.stack([np.add.outer(g, g) / 2] * 3, -1)
        frames = np.stack([np.clip(base + rng.normal(0, 20, base.shape) + 5 * k, 0, 255)
                           for k in range(5)]).astype(np.uint8)
        err = {}
        for crf in (23, 40):
            out = h264_roundtrip(frames, crf, exe)
            assert out.shape == frames.shape and out.dtype == np.uint8, 'H.264 round trip shape'
            err[crf] = np.abs(out.astype(int) - frames).mean()
        assert err[40] > err[23], 'CRF 40 should lose more than CRF 23'
        log(f'PASS: H.264 round trip, 5 frames 224x224x3 uint8; mean abs error CRF23 {err[23]:.1f}, '
            f'CRF40 {err[40]:.1f} ({note})')
    return True


## Cell 5: d1lib, the train-time augmentations and the transform wrapper

In [ ]:
%%writefile d1lib.py
"""D1: train-time codec augmentations for EVA-02 + LoRA (single-view arms).

Every augmentation acts on the PIL face crop before the backbone's training
transform; the evaluation transform is never touched. Masks reuse codec7's drop
tables and its block-DCT math (one 8x8 keep mask per image, shared by every
block and channel, DC never dropped, no rescaling), applied here to the crop at
native resolution in pixel space, because EVA-02 has no frequency stream.

Randomness comes from Python's `random` module unless a generator is passed:
torch reseeds it in every DataLoader worker (base seed + worker id, new base
seed each epoch), so workers never draw identical masks. Self-contained apart
from codec7 (drop_table, h264_roundtrip, jpeg_roundtrip, paired_bootstrap).
Nothing runs on import.
"""
import contextlib
import math
import random
import subprocess

import numpy as np
import pandas as pd
from PIL import Image

import codec7 as C7


MASK_KINDS = C7.MASK_KINDS   # ('qtable', 'uniform', 'inverted')
# Each arm is an ordered list of operations on the crop; the arms differ only in this.
ARM_OPS = {
    'plain':             (),
    'realh264':          ('h264',),
    'qmask':             ('qtable',),
    'randmask':          ('uniform',),
    'realjpeg':          ('jpeg',),
    'randmask_realh264': ('uniform', 'h264'),
    'qmask_realh264':    ('qtable', 'h264'),
    'invmask':           ('inverted',),
    'noaug':             ('identity',),   # retrained without augmentation: the same-session control for plain
}
ARM_ORDER = list(ARM_OPS)
OP_KEYS = {   # config keys each operation depends on (enter the arm's signature)
    'qtable': ('max_drop',), 'uniform': ('max_drop',), 'inverted': ('max_drop',),
    'h264': ('aug_h264_p', 'aug_h264_crf', 'aug_h264_size', 'aug_h264_preset', 'aug_h264_fps'),
    'jpeg': ('aug_jpeg_p', 'aug_jpeg_q', 'jpeg_subsampling'),
    'identity': (),
}


def trained(arm):
    return bool(ARM_OPS[arm])


def arm_params(arm, cfg):
    keys = sorted({k for op in ARM_OPS[arm] for k in OP_KEYS[op]})
    return {k: cfg[k] for k in keys}


# ------------------------------------------------------------------ masks

def dct_basis():
    """Orthonormal 8-point DCT-II matrix, identical to codec7.BlockDCTMask.basis."""
    n = np.arange(8, dtype=np.float64)
    basis = np.cos(math.pi * (2 * n + 1) * n[:, None] / 16) * math.sqrt(2 / 8)
    basis[0] /= math.sqrt(2)
    return basis


BASIS = dct_basis()


def draw_keep(table, rng):
    """One 8x8 keep mask: coefficient (u, v) is dropped with probability table[u, v]."""
    return rng.random((8, 8)) >= table


def mask_array(x, keep):
    """float (h, w, c) -> float (h, w, c). Edge-pad to a multiple of 8, zero the dropped
    coefficients of every 8x8 block of every channel, crop back. Not clipped."""
    h, w, c = x.shape
    x = np.pad(x, ((0, -h % 8), (0, -w % 8), (0, 0)), mode='edge')
    H, W = x.shape[:2]
    blocks = x.reshape(H // 8, 8, W // 8, 8, c).transpose(0, 2, 4, 1, 3)   # (bh, bw, c, 8, 8)
    coeff = BASIS @ blocks @ BASIS.T
    out = BASIS.T @ (coeff * keep) @ BASIS
    return out.transpose(0, 3, 1, 4, 2).reshape(H, W, c)[:h, :w]


def mask_image(image, table, rng):
    x = np.asarray(image.convert('RGB'), dtype=np.float64)
    out = mask_array(x, draw_keep(table, rng))
    return Image.fromarray(np.clip(np.rint(out), 0, 255).astype(np.uint8))


# ------------------------------------------------------------ real codecs

def h264_image(image, crf, exe, size=224, preset='veryfast', fps=25):
    """Resize to size x size (PIL bilinear, as Gate 0's clips), one-frame libx264 clip, decode."""
    a = np.asarray(image.convert('RGB').resize((size, size), Image.Resampling.BILINEAR), dtype=np.uint8)
    try:
        out = C7.h264_roundtrip(a[None], crf, exe, preset, fps, threads=1)
    except (RuntimeError, subprocess.TimeoutExpired):   # one retry: a transient ffmpeg failure would kill a fold
        out = C7.h264_roundtrip(a[None], crf, exe, preset, fps, threads=1)
    return Image.fromarray(out[0])


class CodecAug:
    """Applies `ops` in order to a PIL crop. Mask ops always apply (p = 1); 'h264' and
    'jpeg' apply with their own probability and draw their quality per call; 'identity' does nothing."""

    def __init__(self, ops, max_drop=0.35, exe=None, h264_p=0.5, h264_crf=(23, 40), h264_size=224,
                 h264_preset='veryfast', h264_fps=25, jpeg_p=0.5, jpeg_q=(20, 95), jpeg_subsampling=2):
        unknown = [op for op in ops if op not in MASK_KINDS + ('h264', 'jpeg', 'identity')]
        if unknown or not ops:
            raise ValueError(f'Unknown or empty ops {ops}')
        if 'h264' in ops and exe is None:
            raise ValueError('The h264 op needs a working ffmpeg (codec7.find_ffmpeg)')
        self.ops, self.exe = tuple(ops), exe
        self.tables = {op: C7.drop_table(op, max_drop) for op in ops if op in MASK_KINDS}
        self.h264 = dict(p=h264_p, crf=tuple(h264_crf), size=h264_size, preset=h264_preset, fps=h264_fps)
        self.jpeg = dict(p=jpeg_p, q=tuple(jpeg_q), subsampling=jpeg_subsampling)

    def __call__(self, image, rng=None):
        # Default: the module RNG, which torch reseeds per worker. A Random object stored on
        # self would be copied into every forked worker with the same state.
        rng = random if rng is None else rng
        for op in self.ops:
            if op in self.tables:
                image = mask_image(image, self.tables[op], np.random.default_rng(rng.getrandbits(64)))
            elif op == 'h264':
                if rng.random() < self.h264['p']:
                    image = h264_image(image, rng.randint(*self.h264['crf']), self.exe, self.h264['size'],
                                       self.h264['preset'], self.h264['fps'])
            elif op == 'jpeg' and rng.random() < self.jpeg['p']:
                image = C7.jpeg_roundtrip(image, rng.randint(*self.jpeg['q']), self.jpeg['subsampling'])
        return image

    def __repr__(self):
        return f'CodecAug({self.ops})'


def make_aug(arm, cfg, exe=None):
    """None for 'plain'; otherwise the arm's CodecAug built from the config."""
    ops = ARM_OPS[arm]
    if not ops:
        return None
    return CodecAug(ops, max_drop=cfg['max_drop'], exe=exe, h264_p=cfg['aug_h264_p'],
                    h264_crf=cfg['aug_h264_crf'], h264_size=cfg['aug_h264_size'],
                    h264_preset=cfg['aug_h264_preset'], h264_fps=cfg['aug_h264_fps'],
                    jpeg_p=cfg['aug_jpeg_p'], jpeg_q=cfg['aug_jpeg_q'], jpeg_subsampling=cfg['jpeg_subsampling'])


# -------------------------------------------------------- transform wrap

class AugTransform:
    """aug(PIL) then base_tf; aug None passes the crop straight to base_tf. Top level, so it pickles."""

    def __init__(self, base_tf, aug):
        self.base_tf, self.aug = base_tf, aug

    def __call__(self, image):
        return self.base_tf(image if self.aug is None else self.aug(image))

    def __repr__(self):
        return f'AugTransform({self.aug!r}, {self.base_tf!r})'


@contextlib.contextmanager
def augmented_transforms(namespace, aug):
    """Rebind namespace['backbone_transforms'] so that it returns (AugTransform(tf_train, aug),
    tf_eval, norm); the original is restored on exit, also after an exception. Pass the
    namespace the library functions resolve it in (run_lora_fold.__globals__)."""
    original = namespace['backbone_transforms']
    if getattr(original, 'd1_original', None) is not None:
        raise RuntimeError('backbone_transforms is already wrapped')

    def wrapped(model, size):
        wrapped.calls += 1
        tf_train, tf_eval, norm = original(model, size)
        return AugTransform(tf_train, aug), tf_eval, norm

    wrapped.calls, wrapped.d1_original = 0, original
    namespace['backbone_transforms'] = wrapped
    try:
        yield wrapped
    finally:
        namespace['backbone_transforms'] = original


# ---------------------------------------------------------------- testing

def _synthetic(h, w, seed):
    rng = np.random.default_rng(seed)
    g = np.add.outer(np.linspace(0, 200, h), np.linspace(0, 55, w))
    x = np.stack([g, g[::-1], g[:, ::-1]], -1) + rng.normal(0, 25, (h, w, 3))
    return Image.fromarray(np.clip(x, 0, 255).astype(np.uint8))


def unit_tests(exe=None, log=print):
    """CPU, synthetic images, no downloads. exe: ffmpeg from codec7.find_ffmpeg() or None."""
    ac = np.ones((8, 8), bool)
    ac[0, 0] = False
    tables = {k: C7.drop_table(k, 0.35) for k in MASK_KINDS}
    means = [t[ac].mean() for t in tables.values()]
    assert max(means) - min(means) < 1e-9 and all(t[0, 0] == 0 for t in tables.values()), 'tables'
    assert np.allclose(BASIS @ BASIS.T, np.eye(8)), 'DCT basis not orthonormal'
    rng = np.random.default_rng(0)
    for kind, t in tables.items():
        keep = np.stack([draw_keep(t, rng) for _ in range(20000)])
        assert keep[:, 0, 0].all(), f'{kind}: DC dropped'
        assert np.abs((1 - keep.mean(0)) - t).max() < 0.02, f'{kind}: empirical drop rate differs from the table'
    log(f'PASS: qmask/randmask/invmask tables share mean AC drop {means[0]:.6f}, DC kept, empirical rates match')

    x = np.asarray(_synthetic(64, 48, 1), dtype=np.float64)
    assert np.allclose(mask_array(x, np.ones((8, 8), bool)), x, atol=1e-9), 'keep-all mask is not the identity'
    for kind, t in tables.items():
        y = mask_array(x, draw_keep(t, np.random.default_rng(3)))
        bx = x.reshape(8, 8, 6, 8, 3).mean((1, 3))
        by = y.reshape(8, 8, 6, 8, 3).mean((1, 3))
        assert np.allclose(bx, by, atol=1e-9), f'{kind}: block DC (mean) changed'
        assert ((y - by.repeat(8, 0).repeat(8, 1)) ** 2).sum() <= ((x - bx.repeat(8, 0).repeat(8, 1)) ** 2).sum() + 1e-6, \
            f'{kind}: AC energy increased'
    for h, w in [(128, 128), (131, 97), (160, 160), (7, 13)]:
        img = _synthetic(h, w, h + w)
        same = mask_image(img, np.zeros((8, 8)), np.random.default_rng(0))
        assert np.array_equal(np.asarray(same), np.asarray(img)), f'{h}x{w}: zero table changed the crop'
        for kind, t in tables.items():
            out = mask_image(img, t, np.random.default_rng(5))
            a = np.asarray(out)
            assert out.size == img.size and out.mode == 'RGB' and a.dtype == np.uint8, f'{kind} {h}x{w}: size/type'
            assert a.min() >= 0 and a.max() <= 255
            if min(h, w) >= 8:
                assert not np.array_equal(a, np.asarray(img)), f'{kind} {h}x{w}: crop unchanged'
    log('PASS: masks keep the DC of every block, never add AC energy, keep size and [0, 255], pad/crop non-multiples of 8')

    img = _synthetic(131, 97, 7)
    aug = CodecAug(('qtable',))
    outs = [np.asarray(aug(img)) for _ in range(6)]   # successive samples in one worker
    assert all(not np.array_equal(outs[0], o) for o in outs[1:]), 'identical masks across samples'
    random.seed(1001)
    a = np.asarray(aug(img))
    random.seed(1002)   # what torch does in worker 2: base seed + worker id
    b = np.asarray(aug(img))
    random.seed(1001)
    assert not np.array_equal(a, b) and np.array_equal(a, np.asarray(aug(img))), 'per-worker reseeding'
    r1, r2 = random.Random(9), random.Random(9)
    assert np.array_equal(np.asarray(aug(img, r1)), np.asarray(aug(img, r2))), 'explicit RNG not reproducible'
    log('PASS: masks vary across samples and across workers (module RNG reseeded per worker), reproducible by seed')

    jp = CodecAug(('jpeg',), jpeg_p=1.0, jpeg_q=(20, 20))
    out = jp(img)
    assert out.size == img.size and not np.array_equal(np.asarray(out), np.asarray(img)), 'real JPEG'
    assert np.array_equal(np.asarray(out), np.asarray(C7.jpeg_roundtrip(img, 20))), 'real JPEG differs from codec7'
    never = CodecAug(('jpeg',), jpeg_p=0.0)
    assert never(img) is img, 'p = 0 must pass through'
    r = random.Random(4)
    half = CodecAug(('jpeg',), jpeg_p=0.5, jpeg_q=(20, 95))
    hits = np.mean([half(img, r) is not img for _ in range(400)])
    assert 0.4 < hits < 0.6, f'real-JPEG rate {hits}'
    log(f'PASS: real JPEG at native size (equals codec7.jpeg_roundtrip), applied at rate {hits:.2f} for p = 0.5')

    if exe is None:
        log('SKIP: realh264 augmentation test (no ffmpeg with libx264)')
    else:
        err = {}
        for crf in (23, 40):
            out = CodecAug(('h264',), exe=exe, h264_p=1.0, h264_crf=(crf, crf))(img)
            ref = np.asarray(img.resize((224, 224), Image.Resampling.BILINEAR), dtype=np.int16)
            assert out.size == (224, 224) and out.mode == 'RGB', 'realh264 size'
            err[crf] = np.abs(np.asarray(out, dtype=np.int16) - ref).mean()
        assert err[40] > err[23] > 0, f'CRF 40 must change the crop more than CRF 23: {err}'
        both = CodecAug(('qtable', 'h264'), exe=exe, h264_p=1.0)(img)
        assert both.size == (224, 224)
        log(f'PASS: realh264 one-frame clip 224x224, mean abs change CRF23 {err[23]:.1f}, CRF40 {err[40]:.1f}; '
            f'mask then H.264 composes')

    marker = object()
    assert AugTransform(lambda im: im, None)(marker) is marker, 'AugTransform(None) must pass through'
    seen = []
    t = AugTransform(lambda im: seen.append(im) or 'out', lambda im: ('aug', im))
    assert t('x') == 'out' and seen == [('aug', 'x')], 'AugTransform order'

    def fake_bt(model, size):
        return 'train', 'eval', {'mean': 0}

    ns = {'backbone_transforms': fake_bt}
    try:
        with augmented_transforms(ns, 'AUG') as w:
            tr, ev, _ = ns['backbone_transforms'](None, 224)
            assert isinstance(tr, AugTransform) and tr.aug == 'AUG' and ev == 'eval' and w.calls == 1
            raise KeyError('simulated failure inside training')
    except KeyError:
        pass
    assert ns['backbone_transforms'] is fake_bt, 'backbone_transforms not restored after an exception'
    with augmented_transforms(ns, None):
        try:
            with augmented_transforms(ns, None):
                raise AssertionError('nested wrap accepted')
        except RuntimeError:
            pass
    assert ns['backbone_transforms'] is fake_bt
    log('PASS: AugTransform passes through with aug None; rebinding wraps tf_train only and is restored after an exception')

    assert make_aug('plain', {}) is None and [a for a in ARM_ORDER if not trained(a)] == ['plain']
    assert trained('noaug') and CodecAug(('identity',))(img) is img and arm_params('noaug', {}) == {}
    assert ARM_OPS['qmask_realh264'] == ('qtable', 'h264') and ARM_OPS['randmask_realh264'] == ('uniform', 'h264')

    v = np.random.default_rng(2)
    pred = pd.DataFrame(dict(dataset='dfdc', video_id=[f'v{i}' for i in range(60)],
                             group_id=[f'v{i}' for i in range(60)], label=v.integers(0, 2, 60)))
    pred['probability'] = v.uniform(size=60)
    r = C7.paired_bootstrap(pred, pred.copy(), 200, 42)
    assert r['delta'] == 0 and r['ci_low'] == 0 and r['ci_high'] == 0, 'identical arms must give exactly 0'
    log('PASS: paired bootstrap of identical arms = 0 [0, 0]')
    return True


## Cell 6: configuration, data, Gate 0 subsets, checkpoints and helpers

Set `SMOKE`, `FOLDS` and `ARMS` here. The consistency tolerance and the Gate 0 reference AUCs are fixed here before any result is seen.

In [ ]:
# ===================== D1 CONFIG: codec-aware LoRA training of EVA-02 =====================
import importlib
import multiprocessing
import shutil
import sys
import traceback

SMOKE = True    # True: ~45-60 min check of every arm and condition on tiny data. False: the real run (Commit).
FOLDS = ['dfdc', 'ffpp', 'celebdf']        # held-out corpus of each leave-one-dataset-out fold
ARMS = [                                   # priority order; comment out an arm to switch it off
    'plain',                               # Phase 13 run 2 checkpoints re-scored, no training
    'realh264', 'qmask', 'randmask',       # the gap-closure ratios need these
    'realjpeg', 'randmask_realh264', 'qmask_realh264',
    'invmask',
    'noaug',                               # retrained, no augmentation: measures run-to-run noise against plain
]
BACKBONE = 'eva02_l14'
CONSISTENCY_TOL = 0.002                    # plain must reproduce Gate 0 within this (fixed before any result)
GAP_ARMS, GAP_REFS, GAP_CONDS = ('qmask', 'randmask', 'invmask'), ('realh264', 'realjpeg'), ('h264_crf40', 'jpeg20')
# Gate 0 EVA-02 + LoRA AUCs on the same subsets and conditions (results/gate0_compression_shift/gate0_auc.csv, model eva02_lora)
GATE0_EVA02 = {
    'dfdc': {'clean': 0.829724, 'resize224': 0.831493, 'jpeg50': 0.824935, 'jpeg20': 0.807204, 'h264_crf23': 0.810742, 'h264_crf40': 0.760397},
    'ffpp': {'clean': 0.957328, 'resize224': 0.95644, 'jpeg50': 0.949004, 'jpeg20': 0.927328, 'h264_crf23': 0.946684, 'h264_crf40': 0.866752},
    'celebdf': {'clean': 0.87452, 'resize224': 0.876032, 'jpeg50': 0.818884, 'jpeg20': 0.73998, 'h264_crf23': 0.835452, 'h264_crf40': 0.678916},
}
STORE_SEC_PER_VIDEO = 0.12   # one 8-frame clip at preset medium, 4 workers (results/phase14b_codec_mask_cnn/*/h264_crf*_build.json)
UNIT_OVERHEAD_H = 0.03       # building EVA-02 and loading a checkpoint, per (arm, fold)
SEED_SD = 0.006              # EVA-02 LoRA training-seed SD, DFDC fold, seeds 42/43/44 (results/phase13b_followups)
ROOT_OVERRIDES = {}

assert torch.cuda.is_available(), 'Enable a Kaggle GPU (T4 x2).'
CFG = dict(
    backbone=BACKBONE, smoke=SMOKE, seed=42, data_seed=42, device='cuda:0', amp=True,
    multi_gpu=torch.cuda.device_count() > 1, img_size=224,
    train_per_source=200 if SMOKE else 9000, smoke_videos=12, eval_frames_per_video=8,
    probe_C=[1e-3, 1e-1] if SMOKE else [1e-4, 1e-3, 1e-2, 1e-1, 1.0],
    lora_rank=16, lora_alpha=32, lora_dropout=0.05, lora_targets='qv',
    epochs=1 if SMOKE else 4, batch_size=32, eval_batch_size=64, workers=4,
    lr_lora=2e-4, lr_head=2e-4, weight_decay=1e-4, warmup_frac=0.05, grad_clip=1.0,
    bootstrap=200 if SMOKE else 1000, time_limit_h=11.0, log_every=100,
    # evaluation, identical to Gate 0 (05_compression_shift_gate0.ipynb, Cell 5)
    jpeg=(50, 20), jpeg_subsampling=2, h264_crf=(23, 40), h264_videos_per_class=12 if SMOKE else 500,
    h264_seed=42, h264_size=224, h264_preset='medium', h264_fps=25, h264_workers=4,
    # train-time augmentation (d1lib): masks always, real codecs with p = 0.5
    max_drop=0.35, aug_h264_p=0.5, aug_h264_crf=(23, 40), aug_h264_size=224, aug_h264_preset='veryfast',
    aug_h264_fps=25, aug_jpeg_p=0.5, aug_jpeg_q=(20, 95),
    loader_bench_batches=6 if SMOKE else 16, fail_commit_on_error=SMOKE)
# The loader benchmark runs on an idle machine; during training the main process (DataParallel,
# pin-memory thread) takes about one of the 4 vCPUs from the workers.
LOADER_CONTENTION = (CFG['workers'] - 1) / CFG['workers']
OUT = Path('/kaggle/working/d1' + ('_smoke' if SMOKE else ''))
if SMOKE:   # a smoke re-run must exercise the current code, not reuse units an earlier attempt finished
    shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir(parents=True, exist_ok=True)
CACHE = Path('/tmp/d1_h264')   # decoded H.264 test clips: built once per fold, shared by every arm, deleted at the end
shutil.rmtree(CACHE, ignore_errors=True)
CACHE.mkdir(parents=True)
STORES = {}

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
C7 = importlib.reload(sys.modules['codec7']) if 'codec7' in sys.modules else importlib.import_module('codec7')
D1 = importlib.reload(sys.modules['d1lib']) if 'd1lib' in sys.modules else importlib.import_module('d1lib')
assert set(ARMS) <= set(D1.ARM_OPS), f'Unknown arm in {ARMS}; choose from {D1.ARM_ORDER}'
COMPARISONS = [(a, 'plain') for a in D1.ARM_ORDER if a != 'plain'] + [
    ('qmask', 'randmask'), ('qmask', 'invmask'), ('randmask_realh264', 'realh264'),
    ('qmask_realh264', 'realh264'), ('realh264', 'realjpeg')] + [
    (a, 'noaug') for a in D1.ARM_ORDER if a not in ('plain', 'noaug')]   # same-session control, when it ran


def arm_folds(arm):
    """SMOKE trains on the first fold only (plain scores every fold): every code path, about half the time."""
    return FOLDS[:1] if SMOKE and D1.trained(arm) else FOLDS


write_json = C7.write_json

# ---- data: same parser, seed-42 splits and frames as Phase 13, 13b and Gate 0 ----
for name, path in ROOT_OVERRIDES.items():
    PARSERS[name] = (Path(path), PARSERS[name][1])
print('Image roots:', {name: str(pair[0]) for name, pair in PARSERS.items()})
SOURCE_SPLITS = {name: split_by_video(build_manifest(name), seed=42) for name in CORPORA}
SPLIT_ORIGIN = 'original algorithm replay; assumes unchanged dataset snapshot'
pieces = []
for name in CORPORA:
    for split_name, frame in zip(['train', 'val', 'test'], SOURCE_SPLITS[name]):
        part = frame.copy()
        part['dataset'], part['split'] = name, split_name
        pieces.append(part.rename(columns={'video': 'video_id', 'group': 'group_id'}))
MANIFEST = pd.concat(pieces, ignore_index=True)[['path', 'dataset', 'video_id', 'group_id', 'label', 'split']]
for column in ['video_id', 'group_id']:
    MANIFEST[column] = MANIFEST[column].astype(str)
print(MANIFEST.groupby(['dataset', 'split']).size().rename('frames').to_string())
DATA = validate_manifest(MANIFEST)
SPLITS = {held: fold_frames(DATA, held, CFG) for held in CORPORA}   # all three: the cached probe bank spans them
for held, (tr, va, te) in SPLITS.items():
    print(f'{held:>7} held out | train {len(tr):,} frames | source val {len(va):,} | '
          f'test {len(te):,} frames / {te.video_id.nunique():,} videos')

CONDS = ['clean', 'resize224'] + [f'jpeg{q}' for q in CFG['jpeg']] + [f'h264_crf{c}' for c in CFG['h264_crf']]
SUBSET = {}
for held in FOLDS:   # Gate 0's subsets: up to 500 real + 500 fake held-out videos, seed 42 (DFDC: all 381)
    test = SPLITS[held][2]
    keep = C7.h264_subset(test, CFG['h264_videos_per_class'], CFG['h264_seed'])
    key = test.dataset + '||' + test.video_id
    SUBSET[held] = test[key.isin(set(keep.dataset + '||' + keep.video_id))].reset_index(drop=True)
    v = SUBSET[held].drop_duplicates('video_id')
    print(f'{held:>7} held out | test {test.video_id.nunique():,} videos | Gate 0 subset {len(v):,} videos '
          f'({(v.label == 0).sum()} real, {(v.label == 1).sum()} fake), {len(SUBSET[held]):,} frames')

# ---- inputs: one pruned walk that never lists the face-crop folders ----
IMAGE_ROOTS = {str(PARSERS[n][0]) for n in CORPORA}
WANTED = {'faculty_table_eva02_l14.csv', 'd1_run.json'} | {f'lora_{h}_best.pt' for h in CORPORA}
FOUND = {f: [] for f in WANTED}
for dp, dns, fns in os.walk('/kaggle/input'):
    for f in WANTED.intersection(fns):
        FOUND[f].append(Path(dp))
    if dp in IMAGE_ROOTS or dp.count(os.sep) >= 9 or any(f.lower().endswith(IMG_EXT) for f in fns):
        dns[:] = []
FOUND = {f: sorted(v) for f, v in FOUND.items()}


def phase13_run2(d, held):
    """A lora_<held>_best.pt from Phase 13 run 2 (q,v rank 16, seed 42), not a 13b variant or a D1 arm."""
    if any((q / 'd1_run.json').is_file() for q in [d, *d.parents]):
        return False
    meta = d / f'lora_{held}.json'
    if not meta.is_file():
        return True
    s = json.loads(meta.read_text()).get('signature', {})
    return s.get('lora_targets') == 'qv' and s.get('lora_rank') == 16 and s.get('seed') == 42 and not s.get('smoke')


CKPT = {}
for held in FOLDS:   # the folder holding the Phase 13 table (phase13-eva02-output) first, then phase13-eva02-lora
    dirs = sorted(FOUND[f'lora_{held}_best.pt'], key=lambda d: (not (d / 'faculty_table_eva02_l14.csv').is_file(), str(d)))
    ok = [d for d in dirs if phase13_run2(d, held)]
    if ok:
        CKPT[held] = ok[0] / f'lora_{held}_best.pt'
assert set(FOLDS) <= set(CKPT), f'Phase 13 run 2 checkpoints missing for {sorted(set(FOLDS) - set(CKPT))}: attach phase13-eva02-lora'
SHA = {held: C7.file_sha256(p) for held, p in CKPT.items()}
for held, p in CKPT.items():
    print(f'plain {held}: {p} (sha256 {SHA[held][:12]})')
PROBE_DIRS = [str(d) for d in FOUND['faculty_table_eva02_l14.csv']]
print('EVA-02 features:', 'reused from ' + PROBE_DIRS[0] if PROBE_DIRS else 'not attached, recomputed (~20 min)')
REUSE_ROOTS = [d for d in FOUND['d1_run.json'] if d != OUT]
print('Earlier D1 outputs (finished units reused when the signature matches):', [str(d) for d in REUSE_ROOTS] or 'none')
EXE, FFMPEG_NOTE = C7.find_ffmpeg()
assert EXE, FFMPEG_NOTE   # the test conditions and the realh264 arms need H.264
print('ffmpeg:', FFMPEG_NOTE)
write_json(OUT / 'd1_run.json', dict(smoke=SMOKE, arms=ARMS, folds=FOLDS, started_at_hours=hours()))

# ---- helpers shared by Cells 7 to 10 ----
TRAIN_KEYS = ('seed', 'data_seed', 'smoke', 'smoke_videos', 'train_per_source', 'eval_frames_per_video', 'probe_C',
              'lora_rank', 'lora_alpha', 'lora_dropout', 'lora_targets', 'epochs', 'batch_size', 'lr_lora', 'lr_head',
              'weight_decay', 'warmup_frac', 'grad_clip', 'amp')
EVAL_KEYS = ('smoke', 'smoke_videos', 'eval_frames_per_video', 'jpeg', 'jpeg_subsampling', 'h264_crf',
             'h264_videos_per_class', 'h264_seed', 'h264_size', 'h264_preset', 'h264_fps')


def unit_signature(arm, held):
    s = dict(arm=arm, held=held, ops=list(D1.ARM_OPS[arm]), aug=D1.arm_params(arm, CFG),
             evaluation={k: CFG[k] for k in EVAL_KEYS})
    if D1.trained(arm):
        s['training'] = {k: CFG[k] for k in TRAIN_KEYS}
    else:
        s['checkpoint_sha256'] = SHA[held]
    return jsonable(s)


def pred_path(arm, held, cond, root=None):
    return (root or OUT) / f'pred_{arm}_{held}_{cond}.csv'


def unit_status(arm, held):
    meta = OUT / arm / held / 'done.json'
    return json.loads(meta.read_text()).get('status', '?') if meta.is_file() else 'in_progress'


def find_done(arm, held, sig, copy=True):
    """A complete (arm, fold) here or in an attached earlier D1 output, with the same signature
    and all six prediction files; copied into OUT when it comes from an input."""
    for root in [OUT] + REUSE_ROOTS:
        src = root / arm / held
        try:
            r = json.loads((src / 'done.json').read_text())
        except (OSError, ValueError):
            continue
        preds = [pred_path(arm, held, c, root) for c in CONDS]
        if r.get('status') != 'complete' or r.get('signature') != sig or not all(p.is_file() for p in preds):
            continue
        if root != OUT and copy:
            dst = OUT / arm / held
            dst.mkdir(parents=True, exist_ok=True)
            for f in src.iterdir():
                if f.is_file():
                    shutil.copy2(f, dst / f.name)
            for f in preds:
                shutil.copy2(f, OUT / f.name)
            r['reused_from'] = str(src)
            write_json(dst / 'done.json', r)
        return r
    return None


class Pre224:
    """Resize to the H.264 frame size first, exactly as the clips are made, then the model's transform."""
    def __init__(self, tf):
        self.tf = tf

    def __call__(self, im):
        return self.tf(im.resize((CFG['h264_size'], CFG['h264_size']), Image.Resampling.BILINEAR))


def build_store(held, crf, frames):
    s = CFG['h264_size']
    groups = list(frames.groupby(['dataset', 'video_id'], sort=False).indices.values())
    jobs = [(frames.path.iloc[ix].tolist(), crf, EXE, s, CFG['h264_preset'], CFG['h264_fps']) for ix in groups]
    arr_path = CACHE / f'{held}_crf{crf}.npy'
    arr = np.lib.format.open_memmap(arr_path, mode='w+', dtype=np.uint8, shape=(len(frames), s, s, 3))
    t = time.time()
    with multiprocessing.get_context('fork').Pool(CFG['h264_workers']) as pool:   # workers only run ffmpeg
        for k, (ix, decoded) in enumerate(zip(groups, pool.imap(C7._h264_video, jobs, chunksize=2))):
            arr[ix] = decoded
            if k % 200 == 0:
                log(f'  H.264 CRF {crf} | {held}: video {k + 1}/{len(jobs)}')
    arr.flush()
    del arr
    log(f'  H.264 CRF {crf} | {held}: {len(jobs)} videos in {(time.time() - t) / 60:.1f} min')
    return C7.H264Store(np.load(arr_path, mmap_mode='r'), frames.path.tolist())


def ensure_stores(held):
    t = time.time()
    for crf in CFG['h264_crf']:
        if (held, crf) not in STORES:
            log(f'  /tmp free: {shutil.disk_usage("/tmp").free / 2**30:.1f} GiB before H.264 CRF {crf} | {held}')
            STORES[(held, crf)] = build_store(held, crf, SUBSET[held])
    return time.time() - t


def load_eva(ckpt):
    """Gate 0's EVA-02 loader: Phase 13 detector, LoRA and head weights from a lora_<held>_best.pt."""
    assert getattr(backbone_transforms, 'd1_original', None) is None, 'backbone_transforms is still wrapped'
    det = build_detector('eva02_l14', CFG, 1024)
    state = torch.load(ckpt, map_location='cpu', weights_only=True)
    assert set(state) == {n for n, p in det.named_parameters() if p.requires_grad}, f'{ckpt}: wrong keys'
    _, unexpected = det.load_state_dict(state, strict=False)
    assert not unexpected
    return det, backbone_transforms(det.encoder.backbone, CFG['img_size'])[1]


# measured / projected time of finished units, corrects later projections; real-H.264 arms are
# CPU (ffmpeg) bound and the others GPU bound, so they are corrected separately
RATIOS = {'train': [], 'train_h264': [], 'eval': []}


def train_kind(arm):
    return 'train_h264' if 'h264' in D1.ARM_OPS[arm] else 'train'


def factor(kind):
    return float(np.median(RATIOS[kind])) if RATIOS[kind] else 1.0


def projection(arm, held, built=None):
    """Hours for one (arm, fold): training (GPU or augmented loader, whichever is slower), six
    evaluation conditions, and the fold's H.264 stores if they are not built yet."""
    b = PREFLIGHT['bench']
    built = {h for h in FOLDS if all((h, c) in STORES for c in CFG['h264_crf'])} if built is None else built
    tr, va, te = SPLITS[held]
    p = dict(train_img_s=None, train_raw_h=0.0, train_h=0.0)
    if D1.trained(arm):
        loader_ips = LOADER_IPS.get(arm)
        ips = b['train_img_s'] if loader_ips is None else min(b['train_img_s'], LOADER_CONTENTION * loader_ips)
        raw = project_fold_hours((len(tr), len(va), len(te)), dict(b, train_img_s=ips), CFG) / 1.15
        p.update(train_img_s=ips, train_raw_h=raw, train_h=1.15 * raw * factor(train_kind(arm)))
    p['eval_raw_h'] = len(CONDS) * len(SUBSET[held]) / b['eval_img_s'] / 3600
    p['eval_h'] = 1.15 * p['eval_raw_h'] * factor('eval')
    n_vid = len(SUBSET[held].drop_duplicates(['dataset', 'video_id']))
    p['store_h'] = 0.0 if held in built else 1.15 * len(CFG['h264_crf']) * n_vid * STORE_SEC_PER_VIDEO / 3600
    p['hours'] = p['train_h'] + p['eval_h'] + p['store_h'] + UNIT_OVERHEAD_H
    return p


_AUC = {}


def collect_auc(root=None):
    """Every prediction file of a known (arm, fold, condition) under root, with its AUC."""
    root = root or OUT
    rows = []
    for arm in D1.ARM_ORDER:
        for held in CORPORA:
            for cond in CONDS:
                f = pred_path(arm, held, cond, root)
                if not f.is_file():
                    continue
                key = (str(f), f.stat().st_mtime_ns)
                if key not in _AUC:
                    p = C7.read_pred(f)
                    _AUC[key] = dict(auc=C7.macro_auc(p), videos=len(p), real=int((p.label == 0).sum()),
                                     fake=int((p.label == 1).sum()))
                rows.append(dict(arm=arm, fold=held, condition=cond, **_AUC[key], status=unit_status(arm, held),
                                 file=f.name))
    return pd.DataFrame(rows, columns=['arm', 'fold', 'condition', 'auc', 'videos', 'real', 'fake', 'status', 'file'])


def write_auc():
    a = collect_auc()
    a.to_csv(OUT / 'auc.csv', index=False)
    return a


print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
      '| SMOKE' if SMOKE else '| FULL RUN', '| arms:', ARMS, '| folds:', FOLDS)


## Cell 7: unit tests, real-model preflight and time projection

Every line must read `PASS`. The projection lists each (arm, fold) in run order with its hours, and names the arms that finish in this session.

In [ ]:
# ---- CPU unit tests ----
unit_tests(BACKBONE, CFG)   # Phase 13 library: exact EVA-02 class at depth 2, LoRA wiring, folds, bootstrap (sets CKPT_OK)
D1.unit_tests(EXE, log)     # masks, padding, real JPEG, realh264, AugTransform, rebinding, paired bootstrap

# The rebinding the run uses, in the namespace run_lora_fold resolves backbone_transforms in.
assert run_lora_fold.__globals__ is globals()
_original_bt = backbone_transforms
try:
    with D1.augmented_transforms(run_lora_fold.__globals__, D1.make_aug('qmask', CFG)):
        assert backbone_transforms is not _original_bt
        raise RuntimeError('simulated crash inside run_lora_fold')
except RuntimeError:
    pass
assert backbone_transforms is _original_bt, 'backbone_transforms was not restored'
log('PASS: notebook rebinding of backbone_transforms is restored after an exception')

# ---- real EVA-02 weights: LoRA wiring, memory mode and GPU speed (Phase 13 preflight, no augmentation) ----
TRAINED = [a for a in ARMS if D1.trained(a)]
(OUT / 'preflight').mkdir(exist_ok=True)
PREFLIGHT = preflight(BACKBONE, CFG, SPLITS, FOLDS, list(CORPORA), OUT / 'preflight')

# ---- one augmented training batch per arm through tf_train, forward and backward on the real model ----
dev = CFG['device']
det = build_detector(BACKBONE, CFG, PREFLIGHT['dim'])
det.encoder.amp = False   # fp32 for this check: an fp16 gradient may underflow to exactly zero
with torch.no_grad():
    det.head.weight.normal_(0, 0.05)   # a zero head would give LoRA no gradient
tf_train, tf_eval, _ = backbone_transforms(det.encoder.backbone, CFG['img_size'])
train0 = SPLITS[FOLDS[0]][0]
x = y = loss = None
for arm in TRAINED:
    x, y, _ = next(iter(loader(train0, D1.AugTransform(tf_train, D1.make_aug(arm, CFG, EXE)), CFG, train=True)))
    assert tuple(x.shape) == (CFG['batch_size'], 3, CFG['img_size'], CFG['img_size']) and torch.isfinite(x).all()
    det.train()
    det.zero_grad(set_to_none=True)
    loss = F.binary_cross_entropy_with_logits(det(x[:4].to(dev)), y[:4].float().to(dev))
    loss.backward()
    dead = [a for a in adapters(det) if a.B.grad is None or not a.B.grad.abs().sum() > 0]
    assert torch.isfinite(loss) and not dead, f'{arm}: loss {loss.item()}, {len(dead)} adapters without gradient'
    log(f'PASS: {arm}: augmented batch {tuple(x.shape)}, loss {loss.item():.4f}, '
        f'gradients reach all {len(adapters(det))} adapters')
del det, x, y, loss
free()


def loader_speed(aug, n_batches):
    """Training-loader throughput with the arm's augmentation, worker start-up included (conservative)."""
    t, n = time.time(), 0
    for k, (x, _, _) in enumerate(loader(train0, D1.AugTransform(tf_train, aug), CFG, train=True)):
        n += len(x)
        if k + 1 >= n_batches:
            break
    return n / (time.time() - t)


LOADER_IPS = {'none': loader_speed(None, CFG['loader_bench_batches'])}
for arm in TRAINED:
    LOADER_IPS[arm] = loader_speed(D1.make_aug(arm, CFG, EXE), CFG['loader_bench_batches'])
log('  training loader, img/s with %d workers (idle machine): %s | x %.2f for contention in the projection | '
    'GPU train %.0f img/s, eval %.0f img/s' % (
    CFG['workers'], {k: round(v) for k, v in LOADER_IPS.items()}, LOADER_CONTENTION,
    PREFLIGHT['bench']['train_img_s'], PREFLIGHT['bench']['eval_img_s']))

# ---- D1 time projection: each (arm, fold) in run order, as Cell 9 will decide it ----
rows, now, built = [], hours(), set()
for arm in ARMS:
    for held in arm_folds(arm):
        if find_done(arm, held, unit_signature(arm, held), copy=False):
            rows.append(dict(arm=arm, fold=held, hours=0.0, action='done (reuse)'))
            continue
        p = projection(arm, held, built)
        fits = now + p['hours'] <= CFG['time_limit_h']
        if fits:
            now += p['hours']
            built.add(held)
        rows.append(dict(arm=arm, fold=held, train_img_s=p['train_img_s'], train_h=p['train_h'], eval_h=p['eval_h'],
                         store_h=p['store_h'], hours=p['hours'], finish_h=now if fits else np.nan,
                         action='run' if fits else 'SKIP (time budget)'))
PLAN = pd.DataFrame(rows)
PLAN.to_csv(OUT / 'plan.csv', index=False)
print(PLAN.round(2).to_string(index=False))
whole = [a for a in ARMS if not PLAN[PLAN.arm == a].action.str.startswith('SKIP').any()]
print(f'Projected finish: {now:.2f} h of this session (limit {CFG["time_limit_h"]} h, {hours():.2f} h used so far).')
print(f'Arms that finish on every fold in this session: {whole}')
if len(whole) < len(ARMS):
    print(f'Not in this session: {[a for a in ARMS if a not in whole]}. Run them in a new version with this '
          f"version's output attached: finished units are reused, so remove nothing.")
write_json(OUT / 'preflight_d1.json', dict(bench=PREFLIGHT['bench'], micro_batch=PREFLIGHT['micro_batch'],
                                           loader_img_s=LOADER_IPS, loader_contention=LOADER_CONTENTION,
                                           plan=PLAN.to_dict('records'),
                                           projected_finish_h=now, whole_arms=whole, ffmpeg=FFMPEG_NOTE))
print('READY')


## Cell 8: EVA-02 frozen probes (cached features reused)

In [ ]:
# ---- EVA-02 frozen probes: every trained arm's head starts here (LP-FT), as in Phase 13 run 2 ----
# Unaugmented: run_probes builds its own eval transform, and backbone_transforms is not wrapped here.
PROBES, DIM = {}, PREFLIGHT['dim']
if TRAINED:
    (OUT / 'probe').mkdir(exist_ok=True)
    PROBES, DIM = run_probes(BACKBONE, SPLITS, list(CORPORA), CFG, OUT / 'probe', PROBE_DIRS)
    assert DIM == PREFLIGHT['dim']
    display(pd.read_csv(OUT / 'probe' / f'probe_{BACKBONE}.csv').round(4))
else:
    print('Only plain selected: no training, no probes needed.')


## Cell 9: the run

Arm-major in the priority order of `ARMS`: if time runs out, the late arms are the ones missing. Each (arm, fold) is projected first and skipped with a log line if it would cross 11 h.

In [ ]:
# ---- the run: for each arm (priority order), for each fold: train, then the six Gate 0 conditions ----
CHECKS = {}


def evaluate(arm, held, ckpt):
    """Gate 0's evaluation loop for one checkpoint; every condition is written as soon as it is scored."""
    model, tf = load_eva(ckpt)
    net = nn.DataParallel(model) if CFG['multi_gpu'] else model   # only EVA-02 is ever split across the T4s
    conds = [('clean', dict(preprocess=tf)), ('resize224', dict(preprocess=Pre224(tf)))]
    conds += [(f'jpeg{q}', dict(preprocess=tf, quality=q)) for q in CFG['jpeg']]
    conds += [(f'h264_crf{c}', dict(preprocess=tf, store=STORES[(held, c)])) for c in CFG['h264_crf']]
    aucs = {}
    for cond, kw in conds:
        t = time.time()
        pred = C7.predict(net, SUBSET[held], cfg=CFG, log=log, what=f'{arm}/{held}/{cond}', **kw)
        pred.to_csv(pred_path(arm, held, cond), index=False)
        aucs[cond] = C7.macro_auc(pred)
        write_auc()
        free()
        log(f'{arm} | {held} | {cond}: AUC {aucs[cond]:.4f} on {len(pred)} videos ({time.time() - t:.0f} s)')
    del model, net
    free()
    return aucs


def consistency(held):
    """plain must reproduce Gate 0's EVA-02 AUCs: same checkpoint, frames, subset, conditions and code."""
    for cond in CONDS:
        f = pred_path('plain', held, cond)
        if not f.is_file():
            continue
        auc, ref = C7.macro_auc(C7.read_pred(f)), GATE0_EVA02[held][cond]
        gap = auc - ref
        verdict = ('not comparable (smoke subset)' if SMOKE else
                   'PASS' if abs(gap) <= CONSISTENCY_TOL else f'WARN gap {gap:+.4f}')
        CHECKS[(held, cond)] = dict(fold=held, condition=cond, auc=auc, gate0_auc=ref, gap=gap, verdict=verdict)
        log(f'  consistency plain | {held} | {cond}: {auc:.4f} vs Gate 0 {ref:.4f}: {verdict}')
    pd.DataFrame(CHECKS.values()).to_csv(OUT / 'consistency.csv', index=False)


def run_unit(arm, held):
    d = OUT / arm / held
    d.mkdir(parents=True, exist_ok=True)
    sig = unit_signature(arm, held)
    done = find_done(arm, held, sig)
    if done:
        log(f'{arm} | {held}: already complete ' + (f'(reused from {done["reused_from"]})' if done.get('reused_from') else 'in this output'))
        return done
    p = projection(arm, held)
    if hours() + p['hours'] > CFG['time_limit_h']:
        log(f'SKIP {arm} | {held}: needs ~{p["hours"]:.2f} h, {CFG["time_limit_h"] - hours():.2f} h left. '
            f"Run it in a new version with this version's output attached (finished units are reused).")
        r = dict(status='skipped_time_budget', arm=arm, fold=held, projected=p, at_hours=hours(), signature=sig)
        write_json(d / 'done.json', r)
        return r
    log(f'=== {arm} | {held} held out | augmentation {D1.ARM_OPS[arm] or "none"} | ~{p["hours"]:.2f} h projected ===')
    for f in [x for x in d.iterdir() if x.is_file()] + [pred_path(arm, held, c) for c in CONDS]:
        f.unlink(missing_ok=True)   # leftovers of an unfinished attempt must not count as results
    t0 = time.time()
    if D1.trained(arm):
        cfg_u = dict(CFG, time_limit_h=CFG['time_limit_h'] - p['eval_h'] - p['store_h'])   # leave time to score
        bench_u = dict(PREFLIGHT['bench'], train_img_s=p['train_img_s'])
        with D1.augmented_transforms(run_lora_fold.__globals__, D1.make_aug(arm, CFG, EXE)) as wrap:
            # reuse_dirs=[]: the library's signature does not contain the augmentation, so it cannot tell arms apart
            r = run_lora_fold(BACKBONE, held, SPLITS[held], PROBES[held], DIM, cfg_u, d, bench_u, [])
        if r['status'] not in ('complete', 'truncated'):
            r = dict(status=r['status'], arm=arm, fold=held, projected=p, signature=sig)
            write_json(d / 'done.json', r)
            return r
        assert wrap.calls == 1, 'run_lora_fold did not build its transforms through the D1 wrapper'
        train_sec = time.time() - t0
        RATIOS[train_kind(arm)].append(train_sec / 3600 / p['train_raw_h'])
        training = dict(status=r['status'], best_epoch=r['best_epoch'], source_val_auc=r['val_auc'],
                        full_test_auc=r['test_auc'], train_hours=r['train_hours'], wall_seconds=train_sec)
        ckpt = d / f'lora_{held}_best.pt'
    else:
        training = dict(status='complete', source='Phase 13 run 2 (no training)', checkpoint=str(CKPT[held]),
                        checkpoint_sha256=SHA[held])
        ckpt, train_sec = CKPT[held], 0.0
    store_sec = ensure_stores(held)
    t = time.time()
    aucs = evaluate(arm, held, ckpt)
    eval_sec = time.time() - t
    RATIOS['eval'].append(eval_sec / 3600 / p['eval_raw_h'])
    r = dict(status=training['status'], arm=arm, fold=held, ops=list(D1.ARM_OPS[arm]), training=training,
             auc=aucs, seconds=time.time() - t0, train_seconds=train_sec, store_seconds=store_sec,
             eval_seconds=eval_sec, projected=p, finished_at_hours=hours(), signature=sig)
    write_json(d / 'done.json', r)
    write_auc()
    log(f'  {arm} | {held}: {r["status"]} in {r["seconds"] / 60:.1f} min (projected {p["hours"] * 60:.0f} min)')
    return r


STATUS, ERRORS = [], []   # SMOKE: Cell 10 re-raises the failures after the summary and the zip are written
for arm in ARMS:
    for held in arm_folds(arm):
        try:
            r = run_unit(arm, held)
        except Exception as e:
            ERRORS.append(f'{arm} | {held}:\n{traceback.format_exc()}')
            log(f'FAILED {ERRORS[-1]}')
            r = dict(status='failed', arm=arm, fold=held, error=repr(e))
            write_json(OUT / arm / held / 'done.json', r)
            free()
        if arm == 'plain':
            consistency(held)
        STATUS.append(dict(arm=arm, fold=held, status=r.get('status'), reused_from=r.get('reused_from'),
                           best_epoch=(r.get('training') or {}).get('best_epoch'),
                           minutes=round(r.get('seconds', 0) / 60, 1), error=r.get('error')))
        pd.DataFrame(STATUS).to_csv(OUT / 'status.csv', index=False)
display(pd.DataFrame(STATUS))
if CHECKS:
    print(f'plain against Gate 0 (consistency.csv, PASS within {CONSISTENCY_TOL}):')
    display(pd.DataFrame(CHECKS.values()).round(4))


## Cell 10: summary, paired deltas, gap-closure ratios, plots and download

In [ ]:
# ---- summary: every number below is read back from the pred_*.csv files in OUT ----
import faulthandler
import zipfile
import matplotlib.pyplot as plt
from IPython.display import FileLink, Image as ShowImage, display

KEYS = ['dataset', 'video_id', 'group_id', 'label']


def read(arm, held, cond):
    f = pred_path(arm, held, cond)
    return C7.read_pred(f) if f.is_file() else None


def gap_closure(mask, ref, held, cond, force=False):
    """(AUC mask - AUC plain) / (AUC ref - AUC plain) on identical videos. Defined only when the
    ref - plain interval excludes 0; its interval uses the same cluster resamples for all three arms.
    force (SMOKE only) computes it regardless, so the ratio code runs once before the real run."""
    pm, pr, p0 = read(mask, held, cond), read(ref, held, cond), read('plain', held, cond)
    if pm is None or pr is None or p0 is None:
        return None
    row = dict(fold=held, condition=cond, mask_arm=mask, reference_arm=ref, ratio=np.nan, ratio_lo=np.nan,
               ratio_hi=np.nan, ratio_note='not defined: reference gain interval includes 0')
    try:
        num = C7.paired_bootstrap(pm, p0, CFG['bootstrap'], 42)
        den = C7.paired_bootstrap(pr, p0, CFG['bootstrap'], 42)
    except ValueError as e:
        return dict(row, ratio_note=f'error: {e}')
    row.update(mask_gain=num['delta'], mask_gain_lo=num['ci_low'], mask_gain_hi=num['ci_high'],
               reference_gain=den['delta'], reference_gain_lo=den['ci_low'], reference_gain_hi=den['ci_high'],
               reference_gain_beyond_seed_noise=abs(den['delta']) >= 2 * SEED_SD)
    if not (force or den['ci_low'] > 0 or den['ci_high'] < 0):
        return row
    try:   # pandas MergeError is a ValueError
        d = (p0.merge(pm, on=KEYS, suffixes=('', '_m'), validate='one_to_one')
               .merge(pr, on=KEYS, suffixes=('', '_r'), validate='one_to_one'))
        y = d.label.to_numpy() == 1
        w = C7.cluster_weights(d.dataset.astype(str) + '||' + d.group_id.astype(str), CFG['bootstrap'], 42)
        a0, am, ar = (C7.weighted_auc(y, d[c], w) for c in ('probability', 'probability_m', 'probability_r'))
    except ValueError as e:
        return dict(row, ratio_note=f'error: {e}')
    with np.errstate(divide='ignore', invalid='ignore'):
        boot = (am - a0) / (ar - a0)
        ratio = float(np.float64(num['delta']) / den['delta'])
    boot = boot[np.isfinite(boot)]
    lo, hi = np.quantile(boot, [0.025, 0.975]) if len(boot) >= 0.9 * CFG['bootstrap'] else (np.nan, np.nan)
    row.update(ratio=ratio, ratio_lo=lo, ratio_hi=hi,
               ratio_note='forced (SMOKE path test, not a result)' if force else
               'defined: reference gain interval excludes 0')
    return row


try:   # the metadata and the zip are written even if the summary or a plot fails
    AUC = write_auc()
    if AUC.empty:
        print('No predictions under', OUT)
    else:
        print('AUC by fold, arm and condition (auc.csv; all conditions on the Gate 0 subset)')
        wide = AUC.pivot_table(index=['fold', 'arm'], columns='condition', values='auc')
        wide = wide[[c for c in CONDS if c in wide.columns]]
        wide = wide.reindex(sorted(wide.index, key=lambda k: (FOLDS.index(k[0]) if k[0] in FOLDS else 9,
                                                              D1.ARM_ORDER.index(k[1]))))
        wide.to_csv(OUT / 'auc_wide.csv')
        display(wide.round(4))
        MEAN = (AUC.groupby(['arm', 'condition'])
                .agg(mean_auc=('auc', 'mean'), folds=('fold', 'size'), fold_list=('fold', lambda s: ','.join(s)))
                .reset_index())
        MEAN.to_csv(OUT / 'auc_mean_over_folds.csv', index=False)
        print('Mean over folds, point estimates only (auc_mean_over_folds.csv; check the folds column)')
        mw = MEAN.pivot(index='arm', columns='condition', values='mean_auc')
        display(mw.reindex([a for a in D1.ARM_ORDER if a in mw.index])[[c for c in CONDS if c in mw.columns]].round(4))

        rows = []
        for held in FOLDS:
            log(f'Paired bootstrap | {held} held out')
            for cond in CONDS:
                for a, b in COMPARISONS:
                    pa, pb = read(a, held, cond), read(b, held, cond)
                    if pa is None or pb is None:
                        continue
                    row = dict(fold=held, condition=cond, comparison=f'{a} - {b}', arm_a=a, arm_b=b,
                               status_a=unit_status(a, held), status_b=unit_status(b, held))
                    try:
                        row.update(C7.paired_bootstrap(pa, pb, CFG['bootstrap'], 42))   # AUC(a) - AUC(b)
                    except ValueError as e:
                        row['error'] = str(e)
                    rows.append(row)
        DELTAS = pd.DataFrame(rows)
        if 'ci_low' in DELTAS:
            DELTAS['ci_excludes_0'] = (DELTAS.ci_low > 0) | (DELTAS.ci_high < 0)
            # the bootstrap resamples test videos only; two training runs also differ by seed noise
            DELTAS['beyond_seed_noise'] = DELTAS.delta.abs() >= 2 * SEED_SD
        DELTAS.to_csv(OUT / 'paired_deltas.csv', index=False)
        if 'delta' in DELTAS:
            DMEAN = (DELTAS.dropna(subset=['delta']).groupby(['comparison', 'condition'], sort=False)
                     .agg(mean_delta=('delta', 'mean'), folds=('fold', 'size'),
                          folds_ci_above_0=('ci_low', lambda s: int((s > 0).sum())),
                          folds_ci_below_0=('ci_high', lambda s: int((s < 0).sum())))
                     .reset_index())
            DMEAN.to_csv(OUT / 'paired_deltas_mean.csv', index=False)
            print('Paired deltas at the two key conditions (paired_deltas.csv: every condition; '
                  'delta = AUC(first) - AUC(second) on the same videos, cluster bootstrap 95% interval)')
            print(f'The interval covers test-video sampling only, not training noise (seed SD {SEED_SD}): '
                  f'treat |delta| < {2 * SEED_SD:.3f} as not established even when the interval excludes 0 '
                  f'(beyond_seed_noise). noaug - plain, when run, measures that noise directly.')
            key = DELTAS[DELTAS.condition.isin(GAP_CONDS)]
            display(key[['fold', 'condition', 'comparison', 'delta', 'ci_low', 'ci_high', 'ci_excludes_0',
                         'beyond_seed_noise', 'status_a', 'status_b']].round(4))
            print('Mean paired delta over folds (paired_deltas_mean.csv)')
            display(DMEAN[DMEAN.condition.isin(GAP_CONDS)].round(4))

        rows = [gap_closure(m, ref, held, cond, force=SMOKE) for held in FOLDS for cond in GAP_CONDS
                for ref in GAP_REFS for m in GAP_ARMS]
        GAP = pd.DataFrame([r for r in rows if r is not None])
        GAP.to_csv(OUT / 'gap_closure.csv', index=False)
        if len(GAP):
            print('Gap-closure ratio = (mask arm - plain) / (reference - plain) (gap_closure.csv)')
            for r in GAP.to_dict('records'):
                ratio = (f"{r['ratio']:.2f} [{r['ratio_lo']:.2f}, {r['ratio_hi']:.2f}]" if np.isfinite(r['ratio'])
                         else 'not defined')
                gain = (f" | reference gain {r['reference_gain']:+.4f} [{r['reference_gain_lo']:+.4f}, "
                        f"{r['reference_gain_hi']:+.4f}]" if np.isfinite(r.get('reference_gain', np.nan)) else '')
                print(f"  {r['fold']:>7} | {r['condition']:>10} | ({r['mask_arm']} - plain) / "
                      f"({r['reference_arm']} - plain): {ratio}{gain} | {r['ratio_note']}")

        folds = [h for h in FOLDS if h in set(AUC.fold)]
        fig, axes = plt.subplots(1, len(folds), figsize=(5.5 * len(folds), 4.4), sharey=True, squeeze=False)
        for ax, held in zip(axes[0], folds):
            g = AUC[AUC.fold == held]
            for arm in [a for a in D1.ARM_ORDER if a in set(g.arm)]:
                v = g[g.arm == arm].set_index('condition').auc.reindex(CONDS)
                ax.plot(CONDS, v.values, marker='o', label=arm, ls='--' if arm == 'plain' else '-')
            ax.axhline(0.5, color='grey', lw=0.8, ls=':')
            ax.set(title=f'{held} held out', ylabel='video AUC (Gate 0 subset)')
            ax.tick_params(axis='x', rotation=45)
        axes[0, 0].legend(fontsize=7)
        fig.tight_layout()
        fig.savefig(OUT / 'd1_conditions.png', dpi=150)
        plt.close(fig)
        display(ShowImage(filename=str(OUT / 'd1_conditions.png')))
    print(f'Session wall time {hours():.2f} h')
    if SMOKE:
        print('SMOKE RUN: this only checks the pipeline. These AUCs are not results.')
finally:
    STORES.clear()
    shutil.rmtree(CACHE, ignore_errors=True)
    meta = dict(smoke=SMOKE, arms=ARMS, folds=FOLDS, config=CFG, consistency_tol=CONSISTENCY_TOL,
                seed_sd=SEED_SD, loader_contention=LOADER_CONTENTION, errors=globals().get('ERRORS'),
                gate0_eva02_reference=GATE0_EVA02, comparisons=COMPARISONS, arm_ops=D1.ARM_OPS,
                plain_checkpoints={h: dict(path=str(p), sha256=SHA[h]) for h, p in CKPT.items()},
                probe_dirs=PROBE_DIRS, reuse_roots=[str(d) for d in REUSE_ROOTS], ffmpeg=FFMPEG_NOTE,
                split_origin=SPLIT_ORIGIN, preflight=globals().get('PREFLIGHT'),
                loader_img_s=globals().get('LOADER_IPS'), time_ratios=RATIOS,
                status=globals().get('STATUS'), consistency=list(globals().get('CHECKS', {}).values()),
                torch=torch.__version__, timm=timm.__version__, wall_hours=hours(),
                gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
                note='Celeb-DF and DFDC group_id equals video_id, so their intervals resample videos, not identities.')
    write_json(OUT / 'run_metadata.json', meta)
    faulthandler.cancel_dump_traceback_later()   # finished: disarm the Phase 13 hang watchdog
    ARCHIVE = OUT.with_suffix('.zip')
    with zipfile.ZipFile(ARCHIVE, 'w', zipfile.ZIP_DEFLATED) as z:   # checkpoints and features stay in the version output
        for p in sorted(OUT.rglob('*')):
            if p.is_file() and p.suffix not in ('.pt', '.npy'):
                z.write(p, p.relative_to(OUT.parent))
display(FileLink(os.path.relpath(ARCHIVE, os.getcwd())))
if globals().get('ERRORS'):
    print(f'{len(ERRORS)} unit(s) failed (status.csv, run_metadata.json):')
    print('\n'.join(ERRORS))
    if CFG['fail_commit_on_error']:   # SMOKE: a red commit, but with the summary and the zip written
        raise RuntimeError(f'{len(ERRORS)} unit(s) failed in Cell 9; see the tracebacks above')
